# AGF — Publication figures (qualitative grid + insertion/deletion curves)

Regenerates two figures without built-in titles and with consistent labels.

**Inputs to add (Add Input):**
1. Brain Tumor MRI dataset (same as the main study) — required
2. `benchmarking` notebook output (trained models) — required
3. `agreement-guided-fusion-agf-final` notebook output — required for the curves (`*_aopc_curves.csv`)
4. `agf-tau-selection-and-analyses` notebook output — optional; supplies the frozen τ* (`tau_selection.json`)

**Accelerator:** GPU T4. Then **Run All** (about 15–25 minutes, mostly the setup checks).

Outputs: `/kaggle/working/Figures/qualitative_grid.pdf` and `aopc_curves_<backbone>.pdf`.

In [1]:
# ── §1  FINAL EXPERIMENT CONFIGURATION ─────────────────────────────────────
# Pure Python: no torch import here, so this cell can be read/diffed on its own.
import os

CONFIG_VERSION = "agf-final-1.0"          # stamped into every per-image CSV + manifest

# ── Paths ──────────────────────────────────────────────────────────────────
MODEL_DIR = os.environ.get(
    "BT_MODEL_DIR",
    "/kaggle/input/notebooks/tamannajarintima/benchmarking",   # benchmarking output
)
_DATA_ROOT_CANDIDATES = [
    os.environ.get("BT_DATASET_ROOT", ""),
    "/kaggle/input/brain-tumor-mri-dataset",
    "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset",
]
DATASET_ROOT = next((p for p in _DATA_ROOT_CANDIDATES if p and os.path.exists(p)),
                    _DATA_ROOT_CANDIDATES[-1])
TEST_DIR  = os.path.join(DATASET_ROOT, "Testing")
TRAIN_DIR = os.path.join(DATASET_ROOT, "Training")      # calibration split source
OUTPUT_DIR = os.environ.get("BT_OUTPUT_DIR", "/kaggle/working")
OUTPUT_SUBDIRS = ["Results", "Tables", "Figures", "Statistics", "Logs", "Exports"]

# ── Classification pipeline (READ-ONLY mirror of benchmarking.ipynb) ───────
IMG_SIZE    = 224
NUM_CLASSES = 4
CLASS_NAMES_DEFAULT = ["glioma", "meningioma", "notumor", "pituitary"]
NORM_MEAN   = [0.485, 0.456, 0.406]
NORM_STD    = [0.229, 0.224, 0.225]
HEAD_DROPOUT = 0.4                        # benchmarking DROPOUT_RATE (inactive in eval)
BENCH_RANDOM_SEED = 42                    # benchmarking RANDOM_SEED (train/val split)
BENCH_VAL_FRAC    = 0.15                  # benchmarking VAL_FRAC
# Benchmarking preprocessing cache version this notebook must reproduce:
BENCH_PREPROC_VERSION = "v2:removeblackborder(pad4,thr15,halfopen)+clahe(clip2.0,tile8x8)"
INFERENCE_PRECISION = "fp32"              # XAI model has no autocast; gradients in FP32

MODELS = [
    {"key": "vgg16",       "backbone": "vgg16_bn",    "display": "VGG16-BN",    "ckpt": "vgg16_best.pth"},
    {"key": "densenet121", "backbone": "densenet121", "display": "DenseNet121", "ckpt": "densenet121_best.pth"},
    {"key": "resnet50",    "backbone": "resnet50",    "display": "ResNet50",    "ckpt": "resnet50_best.pth"},
]
RUN_MODELS = ["vgg16", "densenet121", "resnet50"]

# Target layers (resolved and VERIFIED in §7 against the instantiated timm model):
#   vgg16_bn    -> backbone.features      (= output of features[43] MaxPool, 512x7x7;
#                                           features[42] ReLU is 14x14)
#   densenet121 -> backbone.features      (= output of features.norm5 BatchNormAct2d, 1024x7x7)
#   resnet50    -> backbone.layer4        (post-residual, post-ReLU, 2048x7x7)
TARGET_LAYERS = {"vgg16_bn": "features", "densenet121": "features", "resnet50": "layer4"}
TARGET_EXPECTED_HW = (7, 7)

# ── Seeds (every subset has its own, all derived from SEED) ────────────────
SEED            = 42
EVAL_SEED       = SEED          # main evaluation subset
SENS_SEED       = SEED + 1      # robustness subset
ABL_SEED        = SEED + 2      # AGF Full/Equal/Fixed faithfulness ablation subset
WABL_SEED       = SEED + 3      # weighting/kernel/temperature ablation subset
ROBUST_ABL_SEED = SEED + 7      # robustness ablation subset
CORRUPT_SEED    = SEED          # corruption-probe subset
CALIB_SEED      = SEED + 11     # calibration draw from the validation split

# ── Subset sizes ───────────────────────────────────────────────────────────
EVAL_N        = 200             # faithfulness (stratified, 50/class)
GINI_SENS_N   = 50              # robustness (stratified)
ABLATION_N    = 100             # Full/Equal/Fixed faithfulness ablation
CALIB_N       = 100             # AGF-Fixed weight calibration (validation split, NOT test)
PARITY_CHECK_N = None           # None = all 1600 test images for the prediction-parity check
PARITY_WARN_AGREEMENT = 0.995   # clean FP32 vs benchmark clean (AMP) prediction agreement
PARITY_FAIL_AGREEMENT = 0.980   # below this the preprocessing is NOT the benchmark's -> stop

# ── XAI method parameters ──────────────────────────────────────────────────
SG_N_SAMPLES  = 20
SG_NOISE_STD  = 0.10            # only used when SG_NOISE_MODE == "absolute"
SG_NOISE_MODE = "relative"      # sigma = SG_NOISE_FRAC * (max - min) of the normalised input
SG_NOISE_FRAC = 0.15
SG_SEED_MODE  = "content"       # noise seed = hash(image bytes) ^ SEED
IG_N_STEPS    = 30              # trapezoidal rule, 31 evaluations
IG_BASELINE   = "zeros_normalized"   # all-zero tensor in NORMALISED space,
                                     # i.e. the ImageNet-mean colour. The only implemented option.
                                     # NOTE: differs from the AOPC grey baseline; disclosed, not changed.
MAP_NORMALIZATION = "minmax_per_map"          # base maps
MAP_POSTPROC      = "lanczos_resize_224_then_clip_ge0"   # applied identically in every loop
FUSED_NORMALIZATION = "clip_ge0_then_divide_by_max"

# ── AGF ────────────────────────────────────────────────────────────────────
AGREEMENT_KERNEL = "topk_dice"  # the configured AND executed kernel of the v7.1 run
AGREEMENT_TAU    = 1.0
TOPK_FRAC        = 0.10
AGF_WEIGHT_RULE  = "softmax"
KERNEL_ABLATION  = ["ssim_cosine", "pearson", "topk_dice"]

# ── Faithfulness (insertion / deletion) ────────────────────────────────────
AOPC_N_STEPS        = 10
AOPC_PATCH_PCT      = 0.10      # fraction of PIXELS added per step (pixel-level ranking, no patches)
AOPC_BASELINE_PIXEL = 0.5       # neutral grey in pixel space, converted per channel
AOPC_TIE_BREAK_SEED = SEED      # constant tie-break permutation, identical for all methods
AOPC_SCORE          = "softmax_probability"
AOPC_TARGET_CLASS   = "clean_prediction_fixed"       # never re-derived
AOPC_AGGREGATION    = "mean_over_steps_1_to_N"       # step 0 excluded

# ── Robustness (max-sensitivity) ───────────────────────────────────────────
SENS_RADIUS          = 0.02     # per-element Gaussian std in NORMALISED units (not an L2 radius)
SENS_SAMPLES         = 10
SENS_NOISE           = "gaussian_iid_normalized"
SENS_FREEZE_SG_NOISE = True

# ── Ablations (moved here from their cells; values unchanged) ──────────────
ROBUST_ABL_N       = 24
ROBUST_ABL_SAMPLES = 4
PROBE_BACKBONE     = "densenet121"
N_CORRUPT          = 40
CORRUPTIONS        = ["gaussian_noise", "gaussian_blur", "brightness", "contrast"]
SEVERITY = {
    "gaussian_noise": [0.10, 0.25, 0.50, 0.75],
    "gaussian_blur":  [0.50, 1.00, 2.00, 3.00],
    "brightness":     [0.25, 0.50, 0.75, 1.00],
    "contrast":       [0.75, 0.50, 0.35, 0.20],
}
N_ABL                = 40
TEMP_GRID            = [0.25, 0.10, 0.05]
WEIGHT_SPREAD_THRESH = 0.02
QUAL_BACKBONE        = "densenet121"

# ── Statistics ─────────────────────────────────────────────────────────────
ALPHA               = 0.05
STATS_TEST          = "wilcoxon_signed_rank"
STATS_ALTERNATIVE   = "two-sided"
STATS_ZERO_METHOD   = "wilcox"
MULTIPLE_CORRECTION = "holm"
BOOTSTRAP_N         = 1000
# pre-declared hypothesis families. Holm is applied across ALL tests of a family.
PRIMARY_REFERENCE   = "equal"                           # central baseline
PRIMARY_METRICS     = ["aopc_ins", "aopc_del_comp", "sensitivity"]
SECONDARY_BASELINES = ["gradcam_pp", "ig", "smoothgrad"]
METRIC_DIRECTION    = {"aopc_ins": +1, "aopc_del_comp": +1, "aopc_del_raw": -1, "sensitivity": -1}
METRIC_LABEL = {"aopc_ins": "Insertion (AOPC-Ins, higher better)",
                "aopc_del_comp": "Deletion (1 - AOPC-Del, higher better)",
                "aopc_del_raw": "Deletion (raw AOPC-Del, lower better)",
                "sensitivity": "Robustness (max-sensitivity, lower better)"}

# ── Method identity ────────────────────────────────────────────────────────
DISP = {"gradcam_pp": "GradCAM++", "ig": "IG", "smoothgrad": "SmoothGrad",
        "equal": "Equal Fusion", "agf": "AGF (Proposed)"}
ORDER      = ["gradcam_pp", "ig", "smoothgrad", "equal", "agf"]   # Equal Fusion in the main eval
PROPOSED   = "agf"
BASE_KEYS  = ["gradcam_pp", "ig", "smoothgrad"]    # individual attribution methods
FUSION_KEYS = ["equal", "agf"]
BASELINES  = ["equal"] + BASE_KEYS                  # everything AGF is compared against
MCOLOR = {"gradcam_pp": "#E63946", "ig": "#2A9D8F", "smoothgrad": "#457B9D",
          "equal": "#8D99AE", "agf": "#F4A261"}

# ── Consistency guards ─────────────────────────────────────────────────────
assert AGREEMENT_KERNEL in KERNEL_ABLATION
assert IG_BASELINE == "zeros_normalized"
assert set(SECONDARY_BASELINES) == set(BASE_KEYS)
assert PRIMARY_REFERENCE in ORDER and PROPOSED in ORDER
assert all(m in METRIC_DIRECTION for m in PRIMARY_METRICS)
assert PROBE_BACKBONE in {m["key"] for m in MODELS} and QUAL_BACKBONE in {m["key"] for m in MODELS}

_FROZEN_NAMES = [
    "CONFIG_VERSION", "MODEL_DIR", "DATASET_ROOT", "TEST_DIR", "TRAIN_DIR", "OUTPUT_DIR",
    "IMG_SIZE", "NUM_CLASSES", "CLASS_NAMES_DEFAULT", "NORM_MEAN", "NORM_STD", "HEAD_DROPOUT",
    "BENCH_RANDOM_SEED", "BENCH_VAL_FRAC", "BENCH_PREPROC_VERSION", "INFERENCE_PRECISION",
    "RUN_MODELS", "TARGET_LAYERS", "TARGET_EXPECTED_HW",
    "SEED", "EVAL_SEED", "SENS_SEED", "ABL_SEED", "WABL_SEED", "ROBUST_ABL_SEED",
    "CORRUPT_SEED", "CALIB_SEED",
    "EVAL_N", "GINI_SENS_N", "ABLATION_N", "CALIB_N", "PARITY_CHECK_N",
    "PARITY_WARN_AGREEMENT", "PARITY_FAIL_AGREEMENT",
    "SG_N_SAMPLES", "SG_NOISE_STD", "SG_NOISE_MODE", "SG_NOISE_FRAC", "SG_SEED_MODE",
    "IG_N_STEPS", "IG_BASELINE", "MAP_NORMALIZATION", "MAP_POSTPROC", "FUSED_NORMALIZATION",
    "AGREEMENT_KERNEL", "AGREEMENT_TAU", "TOPK_FRAC", "AGF_WEIGHT_RULE", "KERNEL_ABLATION",
    "AOPC_N_STEPS", "AOPC_PATCH_PCT", "AOPC_BASELINE_PIXEL", "AOPC_TIE_BREAK_SEED",
    "AOPC_SCORE", "AOPC_TARGET_CLASS", "AOPC_AGGREGATION",
    "SENS_RADIUS", "SENS_SAMPLES", "SENS_NOISE", "SENS_FREEZE_SG_NOISE",
    "ROBUST_ABL_N", "ROBUST_ABL_SAMPLES", "PROBE_BACKBONE", "N_CORRUPT", "CORRUPTIONS",
    "SEVERITY", "N_ABL", "TEMP_GRID", "WEIGHT_SPREAD_THRESH", "QUAL_BACKBONE",
    "ALPHA", "STATS_TEST", "STATS_ALTERNATIVE", "STATS_ZERO_METHOD", "MULTIPLE_CORRECTION",
    "BOOTSTRAP_N", "PRIMARY_REFERENCE", "PRIMARY_METRICS", "SECONDARY_BASELINES",
    "METRIC_DIRECTION", "ORDER", "PROPOSED", "BASE_KEYS", "FUSION_KEYS", "BASELINES",
]
FROZEN_CONFIG = {k: globals()[k] for k in _FROZEN_NAMES}

print(f"CONFIG {CONFIG_VERSION} | RUN_MODELS={RUN_MODELS}")
print(f"EVAL_N={EVAL_N} GINI_SENS_N={GINI_SENS_N} ABLATION_N={ABLATION_N} CALIB_N={CALIB_N}")
print(f"AGF kernel={AGREEMENT_KERNEL} tau={AGREEMENT_TAU} topk={TOPK_FRAC} | primary reference={DISP[PRIMARY_REFERENCE]}")
print(f"MODEL_DIR={MODEL_DIR}\nTEST_DIR ={TEST_DIR} (exists={os.path.exists(TEST_DIR)})"
      f"\nTRAIN_DIR={TRAIN_DIR} (exists={os.path.exists(TRAIN_DIR)})")

CONFIG agf-final-1.0 | RUN_MODELS=['vgg16', 'densenet121', 'resnet50']
EVAL_N=200 GINI_SENS_N=50 ABLATION_N=100 CALIB_N=100
AGF kernel=topk_dice tau=1.0 topk=0.1 | primary reference=Equal Fusion
MODEL_DIR=/kaggle/input/notebooks/tamannajarintima/benchmarking
TEST_DIR =/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing (exists=True)
TRAIN_DIR=/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training (exists=True)


In [2]:
# ── §2  Environment, imports, determinism ──────────────────────────────────
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # must precede CUDA init

import sys, gc, json, time, glob, random, shutil, warnings, traceback, hashlib, zlib, platform
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from tqdm.auto import tqdm
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import timm

import scipy
from scipy.stats import wilcoxon
import statsmodels
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")


def seed_everything(seed: int = SEED, deterministic: bool = True):
    """Seed Python / NumPy / torch (CPU+CUDA) and pin cuDNN to deterministic kernels.
    Deterministic kernels matter here: the effects under test (e.g. AGF vs Equal)
    are of the order 1e-3, and non-deterministic cuDNN jitter was measured at ~1e-4."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception as _e:
            print(f"[warn] torch.use_deterministic_algorithms unavailable: {_e}")


def stable_seed(*parts) -> int:
    """Process-independent integer seed from arbitrary parts (replaces hash())."""
    return zlib.crc32("|".join(map(str, parts)).encode("utf-8")) & 0x7FFFFFFF


seed_everything(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
ENV_INFO = {
    "python": sys.version.split()[0], "platform": platform.platform(),
    "torch": torch.__version__, "timm": timm.__version__, "numpy": np.__version__,
    "scipy": scipy.__version__, "statsmodels": statsmodels.__version__,
    "opencv": cv2.__version__, "pandas": pd.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cudnn_deterministic": torch.backends.cudnn.deterministic,
    "cudnn_benchmark": torch.backends.cudnn.benchmark,
    "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
    "device": str(DEVICE), "precision": INFERENCE_PRECISION,
}
for k, v in ENV_INFO.items():
    print(f"{k:<24}: {v}")

python                  : 3.13.15
platform                : Linux-6.18.48+-x86_64-with-glibc2.39
torch                   : 2.11.0+cu128
timm                    : 1.0.29
numpy                   : 2.1.3
scipy                   : 1.16.3
statsmodels             : 0.15.0
opencv                  : 4.14.0
pandas                  : 2.3.3
cuda_available          : True
gpu                     : Tesla T4
cudnn_deterministic     : True
cudnn_benchmark         : False
cublas_workspace_config : :4096:8
device                  : cuda:0
precision               : fp32


In [3]:
# ── §3  Output tree, logging, and universal save helpers ───────────────────
# Folders required by the brief. Every artifact is written automatically;
# nothing needs manual saving.
DIRS = {name: os.path.join(OUTPUT_DIR, name) for name in OUTPUT_SUBDIRS}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

_LOG_PATH = os.path.join(DIRS["Logs"], "run.log")
def log(msg: str):
    """Print to console AND append to Logs/run.log with a timestamp."""
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line)
    with open(_LOG_PATH, "a") as f:
        f.write(line + "\n")

def free_gpu(*names, scope=None):
    """Delete named globals if present, then gc + empty CUDA cache.

    `import gc` is LOCAL on purpose. Cells that run their loops at module level
    can rebind and then `del` a global named `gc`, which silently removes the
    module and makes every later free_gpu() raise NameError -- exactly what took
    down Section 19 (and with it Sections 20/14/15) in v7. A local import makes
    this helper immune to whatever the notebook namespace does.
    """
    import gc
    g = scope if scope is not None else globals()
    for n in names:
        if n in g:
            try:    del g[n]
            except Exception: pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── Publication curation: emit ONLY the assets a paper/thesis needs ─────────
# Set PUBLICATION_ONLY = False to restore the full diagnostic figure/table dump.
PUBLICATION_ONLY = True
KEEP_FIGURES = {
    "aopc_curves",                       # per-backbone faithfulness curves
    "cross_arch_aopc_ins",               # cross-architecture faithfulness bar
    "agf_generalization",                # AGF vs best baseline per backbone
    "agf_generalization_forest",         # paired-delta forest plot (preferred)
    "robustness_ablation",               # Full / Equal / Fixed
    "corruption_probe_densenet121",      # faithfulness under corruption
    "weighting_ablation_densenet121",   # weight distribution + temperature
    "qualitative_grid",                  # per-backbone class x method grid
    "qualitative_grid_publication",      # Section 19 class x method overlay grid
    "cross_overall_ranking",             # FIX: the ranking TABLE was whitelisted
                                         # but its figure was silently dropped
}
KEEP_TABLES = {
    "xai_metrics", "statistics", "statistics_robustness", "per_class",  # per-backbone (S1/T3/S2/T5b)
    "cross_classification_results", "cross_xai_aopc_ins",  # T1 / T2
    "cross_overall_ranking", "cross_agf_generalization",   # rank / generalization
    "cross_robustness_sens", "cross_robustness_stats",     # T5 robustness (main body)
    "robustness_ablation_means", "robustness_ablation_stats",     # T4
    "corruption_weighting_densenet121",                   # S3
    "agf_temperature_ablation_densenet121",                # S4
    "agf_weight_diagnostic_densenet121",                   # weight diagnostic
    # FIX: these two are produced and promised in Sections 17/18's markdown
    # but were absent from the whitelist, so save_table() discarded them
    # silently and the deliverable was missing outputs the notebook claimed
    # to have written.
    "corruption_summary_densenet121",                      # Section 17
    "agf_kernel_ablation_densenet121",                 # Section 18
    "runtime", "ablation",                                 # cost + ablation tables
}

# ── v7.1  Per-backbone artifact names, generated for EVERY backbone ────────
#   The literal sets above hard-code "..._densenet121". save_table() and
#   savefig_all() therefore SILENTLY DISCARD the Section 17/18 outputs of any
#   other backbone: a VGG or ResNet run prints its tables, logs success, and
#   writes nothing. The same bug already cost the v7 DenseNet run two
#   artifacts -- there was no entry for `agf_kernel_scores_densenet121` or for
#   the `kernel_ablation_densenet121` FIGURE, so both were computed and thrown
#   away.
#
#   This block is STRICTLY ADDITIVE: every name the literal sets already
#   contain is regenerated identically, so the DenseNet121 curation is
#   unchanged and no previously-emitted artifact can disappear.
for _bb in [_s["key"] for _s in MODELS]:
    KEEP_FIGURES.update({
        f"corruption_probe_{_bb}",          # Section 17
        f"weighting_ablation_{_bb}",        # Section 18 fig 1
        f"kernel_ablation_{_bb}",           # Section 18 fig 2  (was missing)
    })
    KEEP_TABLES.update({
        f"corruption_summary_{_bb}", f"corruption_weighting_{_bb}",     # Section 17
        f"agf_weight_diagnostic_{_bb}", f"agf_temperature_ablation_{_bb}",
        f"agf_kernel_ablation_{_bb}",
        f"agf_kernel_scores_{_bb}",         # Section 18            (was missing)
    })
KEEP_FIGURES.update({"kernel_ablation_cross"})
KEEP_TABLES.update({"agf_kernel_ablation_cross",
                    "agf_temperature_ablation_cross",
                    "agf_weight_diagnostic_cross"})
del _bb

def savefig_all(fig, name: str, subdir: str = None, dpi: int = 300):
    """Save a figure to Figures/ as PNG + PDF + SVG at >=300 DPI, then close it."""
    if PUBLICATION_ONLY and name not in KEEP_FIGURES:
        plt.close(fig); return []
    base = DIRS["Figures"] if subdir is None else os.path.join(DIRS["Figures"], subdir)
    os.makedirs(base, exist_ok=True)
    paths = []
    for ext in ("png", "pdf", "svg"):
        p = os.path.join(base, f"{name}.{ext}")
        fig.savefig(p, dpi=dpi, bbox_inches="tight",
                    facecolor=fig.get_facecolor())
        paths.append(p)
    plt.close(fig)
    del fig
    return paths

def _keep_table(name: str) -> bool:
    """Whitelist test for tables.

    FIX: per-backbone tables are saved as "<Tag>_<base>" (e.g.
    "VGG16BN_statistics") while KEEP_TABLES stores the bare base name
    ("statistics"). A plain `name in KEEP_TABLES` test therefore dropped
    EVERY per-backbone table, and Section 20 later crashed with
    FileNotFoundError when it tried to read them. Accept an exact match or
    a "<prefix>_<base>" suffix match.
    """
    if not PUBLICATION_ONLY:
        return True
    if name in KEEP_TABLES:
        return True
    return any(name.endswith("_" + k) for k in KEEP_TABLES)

def save_table(df: pd.DataFrame, name: str, subdir: str = None,
               float_fmt: str = "%.4f", index: bool = False):
    """Save a DataFrame to Tables/ as CSV + LaTeX (booktabs). Returns paths."""
    if not _keep_table(name):
        return []
    base = DIRS["Tables"] if subdir is None else os.path.join(DIRS["Tables"], subdir)
    os.makedirs(base, exist_ok=True)
    csv_p = os.path.join(base, f"{name}.csv")
    tex_p = os.path.join(base, f"{name}.tex")
    df.to_csv(csv_p, index=index)
    try:
        df.to_latex(tex_p, index=index, float_format=lambda x: float_fmt % x,
                    escape=True, bold_rows=False)
    except Exception as e:
        with open(tex_p, "w") as f:
            f.write(f"% LaTeX export failed: {e}\n")
    return csv_p, tex_p

# One Excel workbook accumulates every table as a sheet (written at the end).
_EXCEL_SHEETS: Dict[str, pd.DataFrame] = {}
def register_excel(sheet_name: str, df: pd.DataFrame):
    """Queue a DataFrame to become a sheet in Exports/all_tables.xlsx."""
    _EXCEL_SHEETS[sheet_name[:31]] = df.copy()      # Excel sheet-name cap = 31 chars

# ── final-version additions to the whitelist (reporting outputs only) ───────
KEEP_FIGURES.update({
    "primary_agf_vs_equal_forest",       # §17 primary-family forest plot
    "cross_arch_metric_means",           # §17 ins / del / sens means per backbone
    "weight_distribution_main",          # §15 main-loop AGF weights
})
KEEP_TABLES.update({
    "stats_primary", "stats_secondary",                  # §15 inferential results
    "per_class_descriptive", "weight_diagnostic_main",   # §15 descriptive
    "cross_metric_means", "calibration_weights",         # §15 / §11
    "prediction_parity", "target_layer_verification",    # §7
})
for _sub in ("calibration", "per_image"):
    os.makedirs(os.path.join(DIRS["Statistics"], _sub), exist_ok=True)

# Section status registry -> written into the experiment manifest (§23).
SECTION_STATUS: Dict[str, str] = {}
def mark_section(name: str, status: str = "completed"):
    SECTION_STATUS[name] = f"{status} @ {time.strftime('%Y-%m-%d %H:%M:%S')}"

log("Output tree ready: " + ", ".join(DIRS.keys()))

[13:07:22] Output tree ready: Results, Tables, Figures, Statistics, Logs, Exports


In [4]:
# ── §4  Preprocessing transforms (verbatim from benchmarking.ipynb §3.1/§3.2) ──
class ApplyCLAHE:
    """CLAHE on the L channel of LAB (stateless; identical to benchmarking.ipynb)."""
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit     = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, pil_img):
        img_np = np.array(pil_img.convert("RGB"))
        lab    = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        clahe  = cv2.createCLAHE(clipLimit=self.clip_limit,
                                 tileGridSize=self.tile_grid_size)
        lab[:, :, 0] = clahe.apply(np.ascontiguousarray(lab[:, :, 0]))
        return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))


class RemoveBlackBorder:
    """Crop the black scanner border (identical to benchmarking.ipynb: half-open
    PIL box with +1, MIN_SIDE guard)."""
    MIN_SIDE = 8

    def __call__(self, img):
        arr  = np.array(img.convert("L"))
        mask = arr > 15
        rows, cols = np.any(mask, axis=1), np.any(mask, axis=0)
        if not rows.any() or not cols.any():
            return img
        r_min, r_max = np.where(rows)[0][[0, -1]]
        c_min, c_max = np.where(cols)[0][[0, -1]]
        pad  = 4
        h, w = arr.shape
        r_min, r_max = max(0, r_min - pad), min(h - 1, r_max + pad)
        c_min, c_max = max(0, c_min - pad), min(w - 1, c_max + pad)
        box = (int(c_min), int(r_min), int(c_max) + 1, int(r_max) + 1)
        if (box[2] - box[0]) < self.MIN_SIDE or (box[3] - box[1]) < self.MIN_SIDE:
            return img
        return img.crop(box)


PREPROC_VERSION = "v2:removeblackborder(pad4,thr15,halfopen)+clahe(clip2.0,tile8x8)"
assert PREPROC_VERSION == BENCH_PREPROC_VERSION, "preprocessing drifted from the benchmark"

val_test_transform = transforms.Compose([
    RemoveBlackBorder(),
    ApplyCLAHE(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])


def load_image_tensor(path: str) -> torch.Tensor:
    """The ONLY image loader used by every XAI loop (PIL convert('RGB') first,
    as ImageFolder's default loader does in the benchmark)."""
    with Image.open(path) as im:
        return val_test_transform(im.convert("RGB"))

print(f"Transforms ready ({PREPROC_VERSION}).")

Transforms ready (v2:removeblackborder(pad4,thr15,halfopen)+clahe(clip2.0,tile8x8)).


In [5]:
# ── §5  Test data, checkpoint discovery, per-model loader ────────────────
if not os.path.exists(TEST_DIR):
    raise FileNotFoundError(
        f"TEST_DIR not found: {TEST_DIR}. Attach 'masoudnickparvar/"
        f"brain-tumor-mri-dataset' or set BT_TEST_DIR.")

test_dataset  = datasets.ImageFolder(TEST_DIR)
sample_paths  = [s[0] for s in test_dataset.samples]
sample_labels = [s[1] for s in test_dataset.samples]
# Stable image ID = "<class>/<file stem>" (extension dropped: the benchmark cache
# stores the same images as .png, ImageFolder order is identical, see §7).
sample_ids    = [os.path.join(os.path.basename(os.path.dirname(p)),
                              os.path.splitext(os.path.basename(p))[0]) for p in sample_paths]

# class names / count: prefer the benchmarking manifest, else ImageFolder order
BENCH_JSON = None
_bench_candidates = [os.path.join(MODEL_DIR, "models", "benchmark_results.json"),
                     os.path.join(MODEL_DIR, "benchmark_results.json")]
_bench_candidates += glob.glob(os.path.join(MODEL_DIR, "**", "benchmark_results.json"),
                               recursive=True)
for _cand in _bench_candidates:
    if os.path.exists(_cand):
        BENCH_JSON = json.load(open(_cand)); break

class_names = list(test_dataset.classes)
assert class_names == CLASS_NAMES_DEFAULT, f"class order {class_names} != benchmark {CLASS_NAMES_DEFAULT}"
num_classes = len(class_names)
assert num_classes == NUM_CLASSES

log(f"Test samples: {len(sample_paths)} | classes={class_names}")
for i, c in enumerate(class_names):
    log(f"    {c}: {sample_labels.count(i)}")

# ── stratified, seeded sampler bound to this dataset ───────────────────────
_cls_to_idxs = defaultdict(list)
for _i, (_, _lbl) in enumerate(test_dataset.samples):
    _cls_to_idxs[int(_lbl)].append(_i)

def stratified_sample(total_n, seed):
    rng = np.random.default_rng(seed)
    base, rem = divmod(total_n, num_classes)
    chosen = []
    for cls_idx in range(num_classes):
        want = base + (1 if cls_idx < rem else 0)
        pool = _cls_to_idxs[cls_idx]
        want = min(want, len(pool))
        chosen.extend(rng.choice(pool, size=want, replace=False).tolist())
    return sorted(chosen)

def class_counts(idxs):
    c = {cn: 0 for cn in class_names}
    for i in idxs:
        c[class_names[sample_labels[i]]] += 1
    return c

# ── checkpoint discovery ───────────────────────────────────────────────────
def find_artifact(fname):
    """Locate a benchmarking artifact under MODEL_DIR (models/ first, then recursive)."""
    for cand in [os.path.join(MODEL_DIR, "models", fname), os.path.join(MODEL_DIR, fname)]:
        if os.path.exists(cand):
            return cand
    hits = sorted(glob.glob(os.path.join(MODEL_DIR, "**", fname), recursive=True))
    return hits[0] if hits else None

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def find_ckpt(spec):
    for cand in [os.path.join(MODEL_DIR, "models", spec["ckpt"]),
                 os.path.join(MODEL_DIR, spec["ckpt"])]:
        if os.path.exists(cand):
            return cand
    hits = glob.glob(os.path.join(MODEL_DIR, "**", spec["ckpt"]), recursive=True)
    return hits[0] if hits else None

print("\nCheckpoint discovery:")
_missing = []
for spec in MODELS:
    p = find_ckpt(spec)
    tag = p if p else "*** NOT FOUND ***"
    print(f"  {spec['display']:<14} -> {tag}")
    if p is None and spec["key"] in RUN_MODELS:
        _missing.append(spec["key"])
if _missing:
    _tree = "\n".join(sorted(glob.glob(os.path.join(MODEL_DIR, "**", "*.pth"),
                                        recursive=True)) or ["  (no .pth found)"])
    raise FileNotFoundError(
        f"Missing checkpoints for {_missing} under MODEL_DIR={MODEL_DIR}.\n"
        f".pth files present:\n{_tree}\n"
        f"Fix MODEL_DIR in Section 2 Config to your benchmarking-output slug.")

# ── per-model loader: reconstruct backbone from checkpoint metadata ────────
def disable_inplace(model):
    """Turn off every in-place activation (ReLU(inplace=True) etc.). VGG16-BN's
    in-place ReLUs modify views that GradCAM++'s full backward hook then tries
    to differentiate, which PyTorch forbids ('...view is being modified
    inplace...'). Non-inplace is numerically identical, just hook-safe."""
    n = 0
    for m in model.modules():
        if getattr(m, "inplace", False):
            m.inplace = False; n += 1
    return n

def load_model(spec):
    """Load one backbone onto DEVICE from its checkpoint. Returns
    (model, eval_model, in_features, backbone_name, ckpt_meta)."""
    path = find_ckpt(spec)
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    assert "model_state_dict" in ckpt, f"{path}: no model_state_dict"
    backbone_name = ckpt.get("backbone_name", spec["backbone"])
    ncls = ckpt.get("num_classes", num_classes)
    # FINAL: checkpoint metadata must match the benchmark contract exactly.
    assert backbone_name == spec["backbone"], f"{path}: backbone {backbone_name} != {spec['backbone']}"
    assert int(ncls) == NUM_CLASSES, f"{path}: num_classes {ncls}"
    if ckpt.get("class_names") is not None:
        assert list(ckpt["class_names"]) == class_names, f"{path}: class_names {ckpt['class_names']}"
    if ckpt.get("img_size") is not None:
        assert int(ckpt["img_size"]) == IMG_SIZE, f"{path}: img_size {ckpt['img_size']}"

    backbone = timm.create_model(backbone_name, pretrained=False,
                                 num_classes=0, global_pool="").to(DEVICE)
    with torch.no_grad():
        dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
        in_features = backbone(dummy).shape[1]
    del dummy
    if ckpt.get("in_features") is not None:
        assert int(ckpt["in_features"]) == int(in_features), \
            f"{path}: in_features {ckpt['in_features']} != rebuilt {in_features}"
    head  = BrainTumorHead(in_features, ncls, dropout=HEAD_DROPOUT).to(DEVICE)
    model = BrainTumorModel(backbone, head).to(DEVICE)
    unwrap(model).load_state_dict(ckpt["model_state_dict"], strict=True)
    disable_inplace(model)          # VGG16-BN uses inplace ReLU -> clashes with GradCAM++ hooks
    model.eval()
    eval_model = unwrap(model); eval_model.eval()
    meta = {k: ckpt.get(k) for k in
            ("backbone_name", "num_classes", "in_features", "img_size", "eval_choice",
             "best_f1", "best_acc", "best_auc", "best_mcc")}
    meta["ckpt_path"] = path
    del ckpt
    free_gpu()
    return model, eval_model, in_features, backbone_name, meta

# ── classification predictions for confusion/ROC/PR (npz reuse or 1 pass) ──
@torch.inference_mode()
def get_predictions(spec, eval_model):
    """Return (y_true, y_pred, y_prob[N,C]) for the classification figures.
    NOTE: when the benchmarking npz is attached these are the benchmark's
    REPORTED TTA x5 predictions (the source of truth for classification
    metrics); they are NOT the clean FP32 predictions that define the XAI
    target class. The latter are recorded per image in the per-image CSVs."""
    key = spec["key"]
    pnpz = os.path.join(MODEL_DIR, "models", f"{key}_preds.npz")
    qnpz = os.path.join(MODEL_DIR, "models", f"{key}_probs.npz")
    if not os.path.exists(pnpz):
        _h = glob.glob(os.path.join(MODEL_DIR, "**", f"{key}_preds.npz"), recursive=True)
        if _h: pnpz = _h[0]
    if not os.path.exists(qnpz):
        _h = glob.glob(os.path.join(MODEL_DIR, "**", f"{key}_probs.npz"), recursive=True)
        if _h: qnpz = _h[0]
    if os.path.exists(pnpz) and os.path.exists(qnpz):
        try:
            pd_ = np.load(pnpz); qd_ = np.load(qnpz)
            def _pick(npz, *names):
                for n in names:
                    if n in npz.files:
                        return npz[n]
                return npz[npz.files[0]]           # fall back to first array
            y_true = _pick(pd_, "true", "y_true", "labels", "targets")
            y_pred = _pick(pd_, "preds", "pred", "y_pred", "predictions")
            y_prob = _pick(qd_, "probs", "prob", "y_prob", "probabilities")
            if len(y_true) == len(y_pred) == len(y_prob):
                return np.asarray(y_true), np.asarray(y_pred), np.asarray(y_prob)
            log(f"  [preds] npz length mismatch for {key}; recomputing.")
        except Exception as e:
            log(f"  [preds] npz load failed for {key} ({e}); recomputing.")
    y_true, y_pred, y_prob = [], [], []
    for path, lbl in tqdm(list(zip(sample_paths, sample_labels)),
                          desc=f"{spec['display']} classify", leave=False):
        t = load_image_tensor(path).unsqueeze(0).to(DEVICE)
        p = torch.softmax(eval_model(t), dim=1)[0].cpu().numpy()
        y_prob.append(p); y_pred.append(int(p.argmax())); y_true.append(int(lbl))
        del t
    return np.array(y_true), np.array(y_pred), np.array(y_prob)

print("\nData + loader ready.")

[13:07:26] Test samples: 1600 | classes=['glioma', 'meningioma', 'notumor', 'pituitary']
[13:07:26]     glioma: 400
[13:07:26]     meningioma: 400
[13:07:26]     notumor: 400
[13:07:26]     pituitary: 400

Checkpoint discovery:
  VGG16-BN       -> /kaggle/input/notebooks/tamannajarintima/benchmarking/models/vgg16_best.pth
  DenseNet121    -> /kaggle/input/notebooks/tamannajarintima/benchmarking/models/densenet121_best.pth
  ResNet50       -> /kaggle/input/notebooks/tamannajarintima/benchmarking/models/resnet50_best.pth

Data + loader ready.


In [6]:
# ── §5b  Calibration split = benchmark validation split (never evaluated on) ──
from sklearn.model_selection import StratifiedShuffleSplit

if not os.path.exists(TRAIN_DIR):
    raise FileNotFoundError(f"TRAIN_DIR not found: {TRAIN_DIR} (needed for the calibration split)")

_train_folder = datasets.ImageFolder(TRAIN_DIR)
assert list(_train_folder.classes) == class_names, "Training/ class order differs from Testing/"
_train_labels = np.asarray(_train_folder.targets)
_sss = StratifiedShuffleSplit(n_splits=1, test_size=BENCH_VAL_FRAC, random_state=BENCH_RANDOM_SEED)
_train_idx, _val_idx = next(_sss.split(np.zeros(len(_train_labels)), _train_labels))
VAL_PATHS  = [_train_folder.samples[i][0] for i in _val_idx]
VAL_LABELS = _train_labels[_val_idx].astype(int)
VAL_IDS    = [os.path.join(os.path.basename(os.path.dirname(p)),
                           os.path.splitext(os.path.basename(p))[0]) for p in VAL_PATHS]
log(f"Benchmark split reconstructed: train={len(_train_idx)} val={len(_val_idx)} "
    f"(benchmark log: 4760 / 840)")
assert len(_train_idx) == 4760 and len(_val_idx) == 840, "split sizes differ from the benchmark"

# Verify against the benchmark's saved validation labels (order-sensitive).
VAL_SPLIT_VERIFIED = None
for _spec in MODELS:
    _p = find_artifact(f"{_spec['key']}_probe.npz")
    if _p:
        _vt = np.load(_p)["val_true"].astype(int)
        VAL_SPLIT_VERIFIED = bool(np.array_equal(_vt, VAL_LABELS))
        assert VAL_SPLIT_VERIFIED, (f"validation label sequence differs from {_p}: the calibration split "
                                    f"is not the benchmark's validation split")
        break
log(f"Validation split label sequence matches benchmark probe: {VAL_SPLIT_VERIFIED} "
    f"(None = probe npz not attached, could not verify)")

# Stratified calibration draw, positions refer to VAL_PATHS.
_rng_cal = np.random.default_rng(CALIB_SEED)
CALIB_POSITIONS = []
for _c in range(num_classes):
    _pool = np.flatnonzero(VAL_LABELS == _c)
    _want = min(len(_pool), CALIB_N // num_classes + (1 if _c < CALIB_N % num_classes else 0))
    CALIB_POSITIONS.extend(_rng_cal.choice(_pool, size=_want, replace=False).tolist())
CALIB_POSITIONS = sorted(CALIB_POSITIONS)
log(f"Calibration images: {len(CALIB_POSITIONS)} from the validation split "
    f"({ {class_names[c]: int((VAL_LABELS[CALIB_POSITIONS] == c).sum()) for c in range(num_classes)} })")

[13:07:38] Benchmark split reconstructed: train=4760 val=840 (benchmark log: 4760 / 840)
[13:07:38] Validation split label sequence matches benchmark probe: True (None = probe npz not attached, could not verify)
[13:07:38] Calibration images: 100 from the validation split ({'glioma': 25, 'meningioma': 25, 'notumor': 25, 'pituitary': 25})


In [7]:
# ── §6  Model architecture (verbatim from benchmarking.ipynb §4) ──────────
# Deliberate difference: no autocast inside BrainTumorModel.forward -> the XAI
# forward/backward passes run in FP32. Weights/architecture are identical.
class BrainTumorHead(nn.Module):
    def __init__(self, in_features, num_classes, dropout=HEAD_DROPOUT):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1 = nn.Linear(in_features * 2, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(256, num_classes)

    def forward(self, x):
        avg = self.avg_pool(x).flatten(1)
        mx  = self.max_pool(x).flatten(1)
        x = torch.cat([avg, mx], dim=1)
        x = F.gelu(self.bn1(self.fc1(x))); x = self.dropout(x)
        x = F.gelu(self.bn2(self.fc2(x))); x = self.dropout(x)
        return self.out(x)


class BrainTumorModel(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head
    def forward(self, x):
        return self.head(self.backbone(x))


def unwrap(model):
    return model.module if isinstance(model, nn.DataParallel) else model

print("Architecture ready.")

Architecture ready.


In [8]:
# ── §7a  Target-layer resolution (used by §7 verification and every loop) ──
# ---- GradCAM++ target layer resolution (backbone-aware) --------------------
def get_target_layer(model, backbone_name: str):
    """Resolve the GradCAM++ target layer: the last SPATIAL, POST-ACTIVATION
    feature map of the backbone.

    WHY THE PREVIOUS VERSION WAS WRONG
    ----------------------------------
    It resolved, per backbone:

        vgg      ->  [m for m in bb.modules() if isinstance(m, Conv2d)][-1]
        densenet ->  bb.features.denseblock4.denselayer16.conv2
        resnet   ->  bb.layer4[-1].conv2
        fallback ->  the last Conv2d found anywhere in the tree

    Each of those is a different failure:

    * VGG16-BN. `convs[-1]` walks the whole module tree in registration
      order, so the last Conv2d is NOT the last conv of `features` -- it is
      `pre_logits.fc2`, a Conv2d(4096, 4096, 1x1) in the classifier head.
      Its activation map is 1x1 SPATIAL. A 1x1 CAM is constant, and the old
      `_normalize` mapped a constant array to exactly 0.0. Result: an
      identically-zero attribution map on 100% of images, which then flowed
      into AOPC (where argsort of zeros is a reverse-raster scan), Gini,
      Entropy, and the AGF weight vector -- where it still received w=0.293.

    * ResNet50. `layer4[-1].conv2` sits BEFORE bn2/act2 in timm's Bottleneck
      (conv2 -> bn2 -> act2), so its output is SIGNED. GradCAM++ derives
      alpha assuming A >= 0; with negative activations `spatial_sum` flips
      the denominator's sign, alpha flips, and np.maximum(cam, 0) zeroes the
      map on roughly 72-79% of images.

    * DenseNet121. `denseblock4.denselayer16.conv2` emits only the 32 NEW
      channels that the last dense layer contributes, not the 1024-channel
      concatenated block output. It does not collapse, but it is not the
      DenseNet feature map either.

    * The Conv2d fallback silently reproduced the VGG failure on any
      unrecognised backbone. It is removed: an unknown backbone now raises.

    Correct targets: bb.features (VGG, 512x7x7), bb.features (DenseNet,
    1024x7x7 via norm5), bb.layer4 (ResNet, 2048x7x7 post-residual).
    """
    bb = unwrap(model).backbone
    name = backbone_name.lower()
    # FINAL: the attribute path comes from CONFIG (TARGET_LAYERS) and is verified
    # structurally in §7. The branches below are the audited v7.1 logic and must
    # agree with CONFIG; a disagreement raises.
    if name in TARGET_LAYERS:
        _cfg_attr = TARGET_LAYERS[name]
        _cfg_mod = bb
        for _part in _cfg_attr.split("."):
            _cfg_mod = getattr(_cfg_mod, _part)
    else:
        raise ValueError(f"backbone '{backbone_name}' has no verified target layer in TARGET_LAYERS")

    if "vgg" in name:
        # timm vgg16_bn: features -> 512 x 7 x 7, post-ReLU/post-pool.
        # NOT convs[-1], which is pre_logits.fc2 (a 1x1 classifier conv).
        assert _cfg_mod is bb.features
        return bb.features

    if "densenet" in name:
        # timm densenet121: features ends in norm5 (BatchNormAct2d, i.e.
        # norm+act fused) -> 1024 x 7 x 7, non-negative.
        assert _cfg_mod is bb.features
        return bb.features

    if "resnet" in name:
        # timm resnet50: layer4 output -> 2048 x 7 x 7, post-residual,
        # post-ReLU. NOT layer4[-1].conv2, which is pre-BatchNorm.
        assert _cfg_mod is bb.layer4
        return bb.layer4

    raise ValueError(
        f"Unresolved target layer for backbone '{backbone_name}'. Add an "
        f"explicit branch to get_target_layer(); the old last-Conv2d "
        f"fallback silently resolved to classifier-head convs and produced "
        f"identically-zero CAMs."
    )


def assert_spatial_target(eval_model, target_layer, img_size=None, device=None,
                          min_hw: int = 7, min_nonneg_frac: float = 0.99,
                          label: str = ""):
    """Probe the resolved target layer once and HARD-FAIL if it cannot
    support GradCAM++. Call immediately after get_target_layer(), before any
    attribution is computed.

    Guards exactly the two failure modes above:
      (1) insufficient spatial extent  -> constant CAM  (the VGG bug)
      (2) signed activations           -> zeroed CAM    (the ResNet bug)

    This is the assertion whose absence let a broken hook produce a full set
    of publication tables and a p = 1.1e-20.
    """
    img_size = IMG_SIZE if img_size is None else img_size
    device   = DEVICE   if device   is None else device
    captured = {}

    def _probe_hook(module, inputs, output):
        if isinstance(output, (tuple, list)):
            output = output[0]
        o = output.detach()
        captured["shape"] = tuple(o.shape)
        captured["min"] = float(o.min())
        captured["nonneg_frac"] = float((o >= 0).float().mean())

    handle = target_layer.register_forward_hook(_probe_hook)
    was_training = eval_model.training
    try:
        eval_model.eval()
        g = torch.Generator(device="cpu").manual_seed(SEED)
        probe_x = torch.randn(1, 3, img_size, img_size, generator=g).to(device)
        with torch.no_grad():
            eval_model(probe_x)
    finally:
        handle.remove()
        if was_training:
            eval_model.train()
        del probe_x

    tag = f" [{label}]" if label else ""
    if "shape" not in captured:
        raise RuntimeError(f"Target layer{tag}: forward hook never fired -- the "
                           f"module is not on this model's forward path.")
    shape = captured["shape"]
    if len(shape) != 4:
        raise RuntimeError(f"Target layer{tag}: output shape {shape}; GradCAM++ "
                           f"requires a 4-D (N, C, H, W) feature map.")
    h, w = shape[-2], shape[-1]
    if h < min_hw or w < min_hw:
        raise RuntimeError(
            f"Target layer{tag}: output {shape} -> spatial extent {h}x{w}, below "
            f"the required {min_hw}x{min_hw}. This is a classifier-head conv, "
            f"not a feature map; the CAM would be constant.")
    if captured["nonneg_frac"] < min_nonneg_frac:
        raise RuntimeError(
            f"Target layer{tag}: only {captured['nonneg_frac']:.1%} of activations "
            f"are non-negative (min={captured['min']:.4f}). GradCAM++ assumes "
            f"A >= 0; hook a post-activation module, not a pre-BatchNorm conv.")
    log(f"[target-layer]{tag} {type(target_layer).__name__} -> {shape} | "
        f"non-negative {captured['nonneg_frac']:.1%} | OK")
    return shape

In [9]:
# ── §7  Verification: checkpoints, preprocessing parity, target layers ─────
@torch.inference_mode()
def _predict_tensors(eval_model, X, bs=32):
    probs = []
    for i in range(0, len(X), bs):
        xb = X[i:i + bs].to(DEVICE)
        probs.append(torch.softmax(eval_model(xb).float(), dim=1).cpu())
    return torch.cat(probs).numpy()

def _hook_shape(eval_model, module, x):
    cap = {}
    h = module.register_forward_hook(lambda m, i, o: cap.__setitem__("s", tuple(o.shape)))
    try:
        with torch.inference_mode():
            eval_model(x)
    finally:
        h.remove()
    return cap.get("s")

_parity_idx = (list(range(len(sample_paths))) if PARITY_CHECK_N is None
               else stratified_sample(PARITY_CHECK_N, SEED))
log(f"[verify] preprocessing {len(_parity_idx)} test + {len(CALIB_POSITIONS)} calibration images once...")
_X_TEST = torch.stack([load_image_tensor(sample_paths[i]) for i in tqdm(_parity_idx, leave=False)])
_X_CAL  = torch.stack([load_image_tensor(VAL_PATHS[p]) for p in tqdm(CALIB_POSITIONS, leave=False)])

PARITY_ROWS, LAYER_ROWS, CKPT_SHA = [], [], {}
for spec in MODELS:
    if spec["key"] not in RUN_MODELS:
        continue
    model, eval_model, in_features, backbone_name, meta = load_model(spec)
    CKPT_SHA[spec["key"]] = sha256_file(meta["ckpt_path"])
    bb = eval_model.backbone
    tl = get_target_layer(eval_model, backbone_name)
    x1 = _X_TEST[:1].to(DEVICE)

    # ---- structural target-layer checks on the instantiated model ----------
    extra = {}
    if backbone_name == "vgg16_bn":
        assert len(bb.features) == 44, f"vgg16_bn.features has {len(bb.features)} modules, expected 44"
        assert isinstance(bb.features[43], nn.MaxPool2d) and isinstance(bb.features[42], nn.ReLU)
        extra["features[42]_shape"] = str(_hook_shape(eval_model, bb.features[42], x1))
        extra["features[43]_shape"] = str(_hook_shape(eval_model, bb.features[43], x1))
        extra["pre_logits_out_shape"] = str(_hook_shape(eval_model, bb.pre_logits, x1))
    elif backbone_name == "densenet121":
        assert list(dict(bb.features.named_children()))[-1] == "norm5"
        extra["norm5_shape"] = str(_hook_shape(eval_model, bb.features.norm5, x1))
    elif backbone_name == "resnet50":
        extra["layer4[-1]_shape"] = str(_hook_shape(eval_model, bb.layer4[-1], x1))
    tl_shape = assert_spatial_target(eval_model, tl, IMG_SIZE, DEVICE, label=spec["display"])
    assert tuple(tl_shape[-2:]) == TARGET_EXPECTED_HW, f"target layer spatial size {tl_shape}"
    head_in = _hook_shape(eval_model, bb, x1)
    LAYER_ROWS.append({"Backbone": spec["display"], "backbone_name": backbone_name,
                       "target_attr": f"backbone.{TARGET_LAYERS[backbone_name]}",
                       "target_module": type(tl).__name__, "target_output_shape": str(tl_shape),
                       "backbone_output_shape(head_input)": str(head_in),
                       "in_features": int(in_features), **extra})
    del x1

    # ---- prediction parity vs the benchmark's saved clean predictions ------
    probe_p = find_artifact(f"{spec['key']}_probe.npz")
    row = {"Backbone": spec["display"], "n_test": len(_parity_idx), "probe_npz": probe_p}
    prob = _predict_tensors(eval_model, _X_TEST)
    pred = prob.argmax(1)
    row["xai_fp32_accuracy_on_checked"] = float((pred == np.asarray(sample_labels)[_parity_idx]).mean())
    if probe_p:
        pz = np.load(probe_p)
        ct = pz["clean_true"].astype(int); cp = pz["clean_pred"].astype(int); cq = pz["clean_prob"]
        assert np.array_equal(ct, np.asarray(sample_labels)), \
            f"{spec['display']}: test label ORDER differs from the benchmark probe"
        row["label_order_identical"] = True
        row["pred_agreement_vs_bench_clean"] = float((pred == cp[_parity_idx]).mean())
        row["n_pred_disagree"] = int((pred != cp[_parity_idx]).sum())
        row["max_abs_prob_diff"] = float(np.abs(prob - cq[_parity_idx]).max())
        row["mean_abs_prob_diff"] = float(np.abs(prob - cq[_parity_idx]).mean())
        cal_prob = _predict_tensors(eval_model, _X_CAL)
        row["calib_pred_agreement_vs_bench_val"] = float(
            (cal_prob.argmax(1) == pz["val_pred"].astype(int)[CALIB_POSITIONS]).mean())
        agr = row["pred_agreement_vs_bench_clean"]
        if agr < PARITY_FAIL_AGREEMENT:
            raise RuntimeError(f"{spec['display']}: prediction agreement {agr:.4f} < "
                               f"{PARITY_FAIL_AGREEMENT}: preprocessing/weights are NOT the benchmark's")
        if agr < PARITY_WARN_AGREEMENT:
            log(f"[verify][WARN] {spec['display']}: agreement {agr:.4f} below {PARITY_WARN_AGREEMENT}")
    else:
        row["label_order_identical"] = None
        log(f"[verify][WARN] {spec['display']}: {spec['key']}_probe.npz not attached -- parity NOT verified")
    PARITY_ROWS.append(row)
    log(f"[verify] {spec['display']}: " + ", ".join(f"{k}={v}" for k, v in row.items() if k != "probe_npz"))
    del model, eval_model, tl, bb
    free_gpu()

del _X_TEST, _X_CAL
PARITY_DF = pd.DataFrame(PARITY_ROWS); LAYER_DF = pd.DataFrame(LAYER_ROWS)
save_table(PARITY_DF, "prediction_parity"); save_table(LAYER_DF, "target_layer_verification")
PARITY_DF.to_csv(os.path.join(DIRS["Statistics"], "prediction_parity.csv"), index=False)
LAYER_DF.to_csv(os.path.join(DIRS["Statistics"], "target_layer_verification.csv"), index=False)
print(PARITY_DF.to_string(index=False)); print(LAYER_DF.to_string(index=False))
mark_section("07_verification")

[13:07:38] [verify] preprocessing 1600 test + 100 calibration images once...


  0%|          | 0/1600 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

[13:08:14] [target-layer] [VGG16-BN] Sequential -> (1, 512, 7, 7) | non-negative 100.0% | OK
[13:08:23] [verify] VGG16-BN: Backbone=VGG16-BN, n_test=1600, xai_fp32_accuracy_on_checked=0.9525, label_order_identical=True, pred_agreement_vs_bench_clean=0.999375, n_pred_disagree=1, max_abs_prob_diff=0.005978494882583618, mean_abs_prob_diff=6.1008991906419396e-05, calib_pred_agreement_vs_bench_val=1.0
[13:08:24] [target-layer] [DenseNet121] Sequential -> (1, 1024, 7, 7) | non-negative 100.0% | OK
[13:08:29] [verify] DenseNet121: Backbone=DenseNet121, n_test=1600, xai_fp32_accuracy_on_checked=0.953125, label_order_identical=True, pred_agreement_vs_bench_clean=0.999375, n_pred_disagree=1, max_abs_prob_diff=0.010363847017288208, mean_abs_prob_diff=0.00034023457556031644, calib_pred_agreement_vs_bench_val=1.0
[13:08:32] [target-layer] [ResNet50] Sequential -> (1, 2048, 7, 7) | non-negative 100.0% | OK
[13:08:37] [verify] ResNet50: Backbone=ResNet50, n_test=1600, xai_fp32_accuracy_on_checked=0.8

In [10]:
%%writefile /kaggle/working/xai_attribution.py
"""
xai_attribution.py  — final-1.0 (algorithmically identical to v7.1)
===================================================================
FINAL-VERSION NOTE: no numerical behaviour of any class in this module was
changed relative to the v7.1 run. Only docstrings that contradicted the code
were corrected (the default kernel, and the definition of max-sensitivity).
Brain Tumor MRI Explainability — pruned XAI core.

Provides:
  BaseAttributor, resize_map, _safe_entropy
  GradCAMPlusPlus, SmoothGrad, IntegratedGradients
  SaliencyMetrics (.sensitivity, .gini)
  AgreementGuidedFusion    (AGF — the proposed method)

v7 change (the substantive one):
  0. The agreement kernel is now PLUGGABLE. [CORRECTED DOCSTRING: the default,
     the notebook CONFIG and the executed v7.1 run all use "topk_dice"; an
     earlier draft of this note said "pearson".] The v6 kernel -- 0.5*(SSIM + raw cosine) on
     min-max-normalised NON-NEGATIVE maps -- is bounded well above zero for
     any two maps sharing a coarse spatial prior. Observed score spread was
     ~0.02, so softmax at tau=1 produced a weight spread of ~0.007 and the
     "agreement-guided" fusion was an arithmetic mean to within 2%. That was
     a property of the kernel, not of brain-tumour MRI. "pearson" (centring)
     and "topk_dice" (thresholding) were both implemented; on the synthetic
     shared-prior control only topk_dice produced usable dynamic range, so it
     was fixed as the kernel BEFORE any test-set evaluation. "ssim_cosine" and
     "pearson" are retained as ablation arms.

v6 changes (correctness, not cosmetics):
  1. BaseAttributor._normalize now RAISES on a degenerate (constant) map
     instead of silently returning all zeros. A constant map is never a
     valid attribution: it means the target layer, class index or gradient
     path is broken. The old silent-zero path let a broken GradCAM++ hook
     propagate through AOPC, Gini, entropy and the fusion weights as if it
     were a real explanation.
  2. SmoothGrad gained freeze_noise(): the max-sensitivity metric compares
     phi(x) against phi(x+delta), and with seed_mode="content" those two
     calls hashed to different seeds and drew independent noise sets. That
     made the robustness column measure Monte-Carlo estimator variance on
     top of input sensitivity for the stochastic methods only, while
     GradCAM++ and IG were deterministic. Freezing the draw across the pair
     makes the comparison apples-to-apples.
  3. SaliencyMetrics.sensitivity returns NaN when every perturbation sample
     is non-finite. It previously returned max_rel_diff == 0.0, i.e. it
     scored a total failure as PERFECT robustness.
  4. AgreementGuidedFusion.fuse_maps() fuses PRE-COMPUTED maps, so the
     evaluation loop no longer computes every base method twice (once
     standalone for its own row, once again inside fuse()). It also
     guarantees the standalone maps and the fusion constituents are the
     identical arrays, which the old path did not: fuse() clipped its
     constituents after resize_map and the standalone arm did not, so Gini
     and Sparsity were computed on subtly different data.

REMOVED in v6: two earlier fusion variants that were retired from the
experiment but still occupied ~250 lines of unreachable code. Recover them
from version control if they are ever re-enabled.
"""

from __future__ import annotations
import hashlib
import numpy as np
import torch
from abc import ABC, abstractmethod
from typing import Dict, Optional, Tuple, List
import cv2

# SSIM is needed by the ssim_cosine ablation kernel. skimage ships on Kaggle,
# but we fall back to a self-contained windowed-SSIM implementation (scipy
# uniform_filter, standard SSIM formula) so this module still works if
# scikit-image is ever missing -- no silent behaviour change either way.
try:
    from skimage.metrics import structural_similarity as _skimage_ssim
    _HAS_SKIMAGE_SSIM = True
except ImportError:
    _HAS_SKIMAGE_SSIM = False
    from scipy.ndimage import uniform_filter

    def _skimage_ssim(a: np.ndarray, b: np.ndarray, data_range: float = 1.0,
                       win_size: int = 7, **_ignored) -> float:
        """Fallback single-scale SSIM (standard formula, uniform window).
        Only used if scikit-image is unavailable. a, b assumed 2-D float32
        in [0, data_range]."""
        C1, C2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
        a, b = a.astype(np.float64), b.astype(np.float64)
        mu_a, mu_b = uniform_filter(a, win_size), uniform_filter(b, win_size)
        mu_a2, mu_b2, mu_ab = mu_a * mu_a, mu_b * mu_b, mu_a * mu_b
        var_a  = uniform_filter(a * a, win_size) - mu_a2
        var_b  = uniform_filter(b * b, win_size) - mu_b2
        cov_ab = uniform_filter(a * b, win_size) - mu_ab
        num = (2 * mu_ab + C1) * (2 * cov_ab + C2)
        den = (mu_a2 + mu_b2 + C1) * (var_a + var_b + C2)
        return float(np.mean(num / (den + 1e-12)))


# ── Utilities ─────────────────────────────────────────────────────────────

def resize_map(attr_map: np.ndarray, size: Tuple[int, int],
               anti_alias: bool = True) -> np.ndarray:
    h, w = size
    interp = cv2.INTER_LANCZOS4 if anti_alias else cv2.INTER_LINEAR
    return cv2.resize(attr_map.astype(np.float32), (w, h), interpolation=interp)


def _safe_entropy(arr: np.ndarray) -> float:
    if float(arr.max()) < 1e-6:
        return 0.0
    p = arr / (arr.sum() + 1e-8)
    return float(-np.sum(p * np.log(p + 1e-8)))


# ── Abstract base ─────────────────────────────────────────────────────────

class BaseAttributor(ABC):
    def __init__(self, model: torch.nn.Module):
        self.model  = model
        self.device = next(model.parameters()).device

    @abstractmethod
    def generate(self, tensor: torch.Tensor,
                 class_idx: Optional[int] = None) -> Tuple[np.ndarray, int]:
        ...

    def __call__(self, tensor: torch.Tensor,
                 class_idx: Optional[int] = None) -> Tuple[np.ndarray, int]:
        return self.generate(tensor, class_idx)

    @staticmethod
    def _normalize(arr: np.ndarray, *, context: str = "") -> np.ndarray:
        """Min-max normalise an attribution map to [0, 1].

        FAILS LOUDLY on degenerate (constant) input. The previous version
        was `arr -= arr.min(); return arr / (arr.max() + 1e-8)`, which maps
        a constant array to EXACTLY zero and returns it without a warning.
        Every downstream consumer -- aopc_curve, gini, entropy, the
        agreement scores, the weighted fusion sum -- then operates on an
        all-zero map as if it were a real explanation. That single silent
        path is how a mis-resolved GradCAM++ target layer (a 1x1 classifier
        conv) produced a full set of publication tables and a p = 1e-20.

        A constant attribution map is never legitimate. It means one of:
          * the target layer has no spatial extent (classifier head conv),
          * the target layer emits signed activations and GradCAM++'s
            np.maximum(cam, 0) zeroed everything,
          * the class index is wrong, or gradients never reached the hook.
        All of those are bugs, so this raises and lets the caller's
        try/except log a SKIP rather than fabricating a data point.

        `context` is a free-text tag (e.g. "gradcam_pp/vgg16_bn") echoed in
        the error message. It is keyword-only with a default, so existing
        `self._normalize(x)` call sites keep working unchanged.
        """
        arr = np.asarray(arr, dtype=np.float64)
        tag = f" [{context}]" if context else ""
        if not np.isfinite(arr).all():
            n_bad = int(np.count_nonzero(~np.isfinite(arr)))
            raise ValueError(
                f"Non-finite attribution map{tag}: {n_bad} of {arr.size} "
                f"entries are NaN or Inf."
            )
        lo, hi = float(arr.min()), float(arr.max())
        if hi - lo < 1e-12:
            raise ValueError(
                f"Degenerate attribution map{tag}: constant value {lo:.6g} "
                f"(dynamic range {hi - lo:.3g}). Check the target layer "
                f"(spatial extent AND non-negativity), the class index, and "
                f"that gradients reach the hooked module."
            )
        return (arr - lo) / (hi - lo)

    def _resolve_class(self, inp: torch.Tensor,
                       class_idx: Optional[int]) -> int:
        if class_idx is not None:
            return class_idx
        with torch.no_grad():
            return int(self.model(inp).argmax(dim=1).item())


# ── GradCAM++ ─────────────────────────────────────────────────────────────

class GradCAMPlusPlus(BaseAttributor):
    def __init__(self, model: torch.nn.Module, target_layer: torch.nn.Module):
        super().__init__(model)
        self._acts = self._grads = None
        self._fwd_hook = target_layer.register_forward_hook(self._cache_acts)
        self._bwd_hook = target_layer.register_full_backward_hook(self._cache_grads)

    def _cache_acts(self, _, __, output):
        self._acts = output.detach()

    def _cache_grads(self, _, __, grad_output):
        self._grads = grad_output[0].detach()

    def __enter__(self): return self
    def __exit__(self, *_): self.remove()

    def remove(self):
        self._fwd_hook.remove()
        self._bwd_hook.remove()

    def generate(self, tensor: torch.Tensor,
                 class_idx: Optional[int] = None) -> Tuple[np.ndarray, int]:
        self.model.eval()
        inp       = tensor.unsqueeze(0).to(self.device).requires_grad_(True)
        out       = self.model(inp)
        class_idx = self._resolve_class(inp, class_idx)
        self.model.zero_grad()
        out[0, class_idx].backward()

        grads = self._grads[0].cpu().numpy()
        acts  = self._acts[0].cpu().numpy()
        g2, g3     = grads ** 2, grads ** 3
        spatial_sum = acts.sum(axis=(1, 2), keepdims=True)
        denom  = 2.0 * g2 + spatial_sum * g3
        denom  = np.where(np.abs(denom) > 1e-8, denom, 1e-8)
        alpha  = g2 / denom
        weights = (alpha * np.maximum(grads, 0)).sum(axis=(1, 2))
        cam = np.einsum("c,chw->hw", weights, acts)
        cam = np.maximum(cam, 0)
        return self._normalize(cam, context="gradcam_pp"), class_idx


# ── SmoothGrad ────────────────────────────────────────────────────────────

class SmoothGrad(BaseAttributor):
    """SmoothGrad with reproducible, arm-independent noise.

    seed_mode="content": the generator is reseeded per call from a hash of
    the input tensor's bytes. Consequences that matter for paired testing:
      * the same image draws the same noise whether SmoothGrad is called
        standalone or from inside AGF, so the AGF-vs-SmoothGrad comparison
        is not confounded by a noise mismatch;
      * different images draw different noise, so the fusion arms are not
        reusing one fixed noise set for all N images;
      * results do not depend on call order, so RUN_MODELS staging and
        re-runs reproduce exactly.
    seed_mode="stream": legacy single advancing generator.

    noise_mode="relative": sigma = noise_frac * (input max - min), i.e. the
    fraction-of-range convention SmoothGrad actually specifies.
    noise_mode="absolute": sigma = noise_std, interpreted in whatever space
    the input already lives in (here: ImageNet-normalised).

    NOISE FREEZING (v6) -- required for an honest max-sensitivity metric
    --------------------------------------------------------------------
    Max-sensitivity evaluates  ||phi(x + delta) - phi(x)|| / ||phi(x)||.
    Under seed_mode="content", x and x+delta hash to DIFFERENT seeds, so the
    two calls drew entirely independent noise sets. The measured quantity
    was therefore

        input sensitivity  +  Monte-Carlo estimator variance

    for SmoothGrad and for any fusion containing it, while GradCAM++ and IG
    are deterministic and contributed only the first term. Methods were
    being compared on different measurands.

    freeze_noise(True) caches the noise draw from the next generate() call
    and reuses it for every subsequent call until freeze_noise(False). Wrap
    the (base, perturbed) pair in a freeze and the estimator variance
    cancels, leaving input sensitivity alone. The cache is keyed on tensor
    SHAPE only -- freezing across differently-shaped inputs is a usage
    error and raises rather than silently re-drawing.
    """

    def __init__(self, model: torch.nn.Module, n_samples: int = 50,
                 noise_std: float = 0.10, squared: bool = False,
                 seed: Optional[int] = None,
                 noise_mode: str = "absolute", noise_frac: float = 0.15,
                 seed_mode: str = "content"):
        super().__init__(model)
        self.n_samples  = n_samples
        self.noise_std  = noise_std
        self.squared    = squared
        self.noise_mode = noise_mode
        self.noise_frac = noise_frac
        self.seed_mode  = seed_mode
        self._base_seed = 0 if seed is None else int(seed)
        self._rng = torch.Generator(device="cpu")
        if seed is not None:
            self._rng.manual_seed(self._base_seed)
        # --- noise-freezing state (v6) ---
        self._freeze          = False
        self._frozen_noise    = None      # list[Tensor] on CPU, len == n_samples
        self._frozen_shape    = None
        self._frozen_sigma    = None

    def freeze_noise(self, enable: bool = True) -> "SmoothGrad":
        """Enable/disable noise freezing. Enabling CLEARS any existing cache,
        so the next generate() call defines the frozen draw. Returns self so
        it can be chained: sg.freeze_noise(True).generate(x, c)."""
        self._freeze = bool(enable)
        self._frozen_noise = None
        self._frozen_shape = None
        self._frozen_sigma = None
        return self

    def _content_seed(self, inp: torch.Tensor) -> int:
        """Stable 63-bit seed from the input bytes, mixed with the base seed."""
        buf = inp.detach().to("cpu", torch.float32).contiguous().numpy().tobytes()
        h   = hashlib.blake2b(buf, digest_size=8,
                              person=b"smoothgr").digest()
        return (int.from_bytes(h, "big") ^ (self._base_seed * 0x9E3779B97F4A7C15)) \
               % ((1 << 63) - 1)

    def _draw_noise(self, inp: torch.Tensor, sigma: float) -> List[torch.Tensor]:
        """Return the n_samples noise tensors for this call, honouring the
        freeze cache. Noise is generated on CPU with self._rng (so it is
        reproducible independently of the CUDA RNG) and moved to device."""
        shape = tuple(inp.shape)
        if self._freeze and self._frozen_noise is not None:
            if self._frozen_shape != shape:
                raise ValueError(
                    f"SmoothGrad noise is frozen for shape {self._frozen_shape} "
                    f"but was called with {shape}. Call freeze_noise(True) again "
                    f"to re-arm, or freeze_noise(False) to disable."
                )
            # Reuse the FROZEN sigma as well as the frozen draw. Under
            # noise_mode="relative" sigma is derived from the input's own
            # range, which shifts slightly under perturbation; holding both
            # the draw and its scale fixed is what makes phi(x) and
            # phi(x+delta) differ only by delta.
            return [n.to(inp.device) * self._frozen_sigma
                    for n in self._frozen_noise]
        # Unit-variance draws are cached; the sigma scaling is applied on
        # the way out so the cache stays independent of the noise mode.
        draws = [torch.randn(shape, generator=self._rng)
                 for _ in range(self.n_samples)]
        if self._freeze:
            self._frozen_noise = [d.clone() for d in draws]
            self._frozen_shape = shape
            self._frozen_sigma = float(sigma)
        return [d.to(inp.device) * sigma for d in draws]

    def generate(self, tensor: torch.Tensor,
                 class_idx: Optional[int] = None) -> Tuple[np.ndarray, int]:
        self.model.eval()
        inp       = tensor.unsqueeze(0).to(self.device)
        class_idx = self._resolve_class(inp, class_idx)
        grad_acc  = torch.zeros_like(inp)
        # Reseeding is skipped while frozen: the cached draw already fixes
        # the noise, and reseeding from the PERTURBED input's bytes is
        # exactly the behaviour freezing exists to suppress.
        if self.seed_mode == "content" and not (self._freeze and
                                                self._frozen_noise is not None):
            self._rng.manual_seed(self._content_seed(inp))
        if self.noise_mode == "relative":
            span  = float(inp.max().item() - inp.min().item())
            sigma = self.noise_frac * span
        else:
            sigma = self.noise_std
        for noise in self._draw_noise(inp, sigma):
            noisy = (inp + noise).requires_grad_(True)
            out   = self.model(noisy)
            self.model.zero_grad()
            out[0, class_idx].backward()
            if noisy.grad is not None:
                g = noisy.grad.detach()
                grad_acc += (g ** 2) if self.squared else g.abs()
            del noise, noisy, out
        saliency = grad_acc[0].mean(dim=0).cpu().numpy()
        return self._normalize(saliency, context="smoothgrad"), class_idx

# ── Integrated Gradients ──────────────────────────────────────────────────

class IntegratedGradients(BaseAttributor):
    def __init__(self, model: torch.nn.Module, n_steps: int = 50):
        super().__init__(model)
        self.n_steps = n_steps

    def generate(self, tensor: torch.Tensor,
                 class_idx: Optional[int] = None,
                 baseline: Optional[torch.Tensor] = None) -> Tuple[np.ndarray, int]:
        self.model.eval()
        inp      = tensor.unsqueeze(0).to(self.device)
        baseline = (torch.zeros_like(inp) if baseline is None
                    else baseline.to(self.device))
        class_idx = self._resolve_class(inp, class_idx)
        grad_sum  = torch.zeros_like(inp)
        for k in range(self.n_steps + 1):
            alpha  = k / self.n_steps
            interp = (baseline + alpha * (inp - baseline)).requires_grad_(True)
            out    = self.model(interp)
            self.model.zero_grad()
            out[0, class_idx].backward()
            if interp.grad is not None:
                w = 0.5 if k in (0, self.n_steps) else 1.0
                grad_sum += w * interp.grad.detach()
        ig     = (inp - baseline) * grad_sum / self.n_steps
        ig_map = ig[0].abs().mean(dim=0).cpu().numpy()
        return self._normalize(ig_map, context="ig"), class_idx


# ── SaliencyMetrics ───────────────────────────────────────────────────────

class SaliencyMetrics:
    def __init__(self, model: torch.nn.Module):
        self.model  = model
        self.device = next(model.parameters()).device

    # NOTE (v08): the old SaliencyMetrics.aopc() method was removed. It was
    # dead code (never called; the evaluation loop uses _aopc_curve in section 4,
    # which correctly uses a grey baseline + softmax + insertion/deletion).
    # The old method masked with raw logits and no grey baseline, so keeping it
    # risked being mistaken for the metric actually reported in Table 1.

    def sensitivity(self, tensor: torch.Tensor, attributor,
                    class_idx: int, radius: float = 0.02,
                    n_samples: int = 20, seed: Optional[int] = None,
                    freeze_stochastic: bool = True) -> float:
        """Relative max-sensitivity, adapted from Yeh et al. 2019.

            Sens_rel = max_{s=1..S} ||phi(x+d_s) - phi(x)||_2 / (||phi(x)||_2 + eps),
            d_s ~ N(0, radius^2 I) i.i.d. per element in NORMALISED input space.

        [CORRECTED DOCSTRING] This is a Monte-Carlo maximum over S Gaussian
        perturbations with per-element std `radius`; it is NOT a supremum over
        an L2 ball of radius r (Yeh et al. sample uniformly in an L-inf ball).
        phi(.) is whatever the attributor returns; the notebook wraps every
        method so that phi is the post-processed 224x224 map (min-max ->
        Lanczos resize -> clip >= 0), identical to what the faithfulness
        metrics consume. The class index is fixed by the caller.

        Dividing by the base-map norm makes the score comparable across
        methods whose maps live on different numeric scales; the raw L2
        version ranked methods by MAP MAGNITUDE, not by robustness.

        v6 FIX 1 -- all-NaN case. The loop below only updates max_rel_diff
        on finite samples, and max_rel_diff starts at 0.0. If EVERY sample
        was non-finite the function returned 0.0, i.e. it reported PERFECT
        robustness -- the best possible score -- for a method that failed
        completely on that image. It now returns NaN so the image is
        excluded from the paired comparison instead of silently winning it.

        v6 FIX 2 -- stochastic/deterministic asymmetry. If `attributor`
        exposes freeze_noise() (SmoothGrad, or a fusion that forwards it),
        its noise draw is frozen across the whole (base, perturbed) family.
        Without this, phi(x) and phi(x+d) drew independent SmoothGrad noise
        and the metric measured input sensitivity PLUS Monte-Carlo
        estimator variance -- but only for the stochastic methods, while
        GradCAM++ and IG contributed input sensitivity alone. Set
        freeze_stochastic=False to reproduce the pre-v6 (confounded)
        numbers.

        NOTE: absolute values are not comparable to any pre-v08 run.
        """
        if seed is not None:
            torch.manual_seed(seed)

        # --- freeze stochastic estimators across the whole pair family ----
        _froze = False
        if freeze_stochastic and hasattr(attributor, "freeze_noise"):
            try:
                attributor.freeze_noise(True)
                _froze = True
            except Exception:
                _froze = False
        try:
            base_attr, _ = attributor(tensor, class_idx)
            base_l2      = float(np.linalg.norm(base_attr))
            # A near-zero / degenerate base map drives the relative
            # denominator toward 0 and the ratio explodes (observed pre-fix:
            # ResNet50 GradCAM++ ~8.7e7). Such a map carries no meaningful
            # stability to measure. NOTE: after the v6 target-layer fix this
            # branch should essentially never fire -- if it does, the target
            # layer is still wrong, so it is logged rather than swallowed.
            if base_l2 < 1e-6:
                print("  [WARN] sensitivity(): degenerate base map "
                      f"(L2={base_l2:.3g}) -- image excluded. Check "
                      "get_target_layer / assert_spatial_target.")
                return float("nan")
            base_norm    = base_l2 + 1e-8   # scale reference
            max_rel_diff = 0.0
            n_nan        = 0
            for _ in range(n_samples):
                noise   = torch.randn_like(tensor) * radius
                pert, _ = attributor(tensor + noise, class_idx)
                diff    = float(np.linalg.norm(pert - base_attr)) / base_norm
                # NaN comparisons are always False in Python, so a NaN diff
                # used to silently freeze the max at 0.0 forever.
                if np.isnan(diff):
                    n_nan += 1
                    continue
                if diff > max_rel_diff:
                    max_rel_diff = diff
                del noise, pert
        finally:
            if _froze:
                try:    attributor.freeze_noise(False)
                except Exception: pass

        if n_nan >= n_samples:
            # Every sample failed: returning max_rel_diff (== 0.0) here would
            # score total failure as perfect robustness. Return NaN and let
            # the nan-aware aggregate exclude the image.
            print(f"  [WARN] sensitivity(): all {n_samples} perturbation "
                  "samples were non-finite -- returning NaN (image excluded).")
            return float("nan")
        if n_nan > 0:
            print(f"  [WARN] sensitivity(): {n_nan}/{n_samples} NaN diffs skipped; "
                  f"max taken over the remaining {n_samples - n_nan}.")
        return float(max_rel_diff)

    @staticmethod
    def gini(attr_map: np.ndarray) -> float:
        arr = np.abs(attr_map).flatten().astype(np.float64)
        if arr.sum() < 1e-6:
            return 0.0
        arr = np.sort(arr)
        n   = len(arr)
        idx = np.arange(1, n + 1, dtype=np.float64)
        num = 2.0 * float((idx * arr).sum())
        den = float(n * arr.sum()) + 1e-8
        g   = num / den - (n + 1.0) / n
        return float(max(0.0, g))


# ── ★ AGF — Agreement-Guided Fusion (proposed method) ──────────────────────

# Defaults; Section 2 CONFIG overrides these on the imported module so the
# kernel is set in exactly one place.
_AGF_DEFAULT_KERNEL = "topk_dice"
_AGF_DEFAULT_TAU    = 1.0
_AGF_DEFAULT_TOPK   = 0.10


class AgreementGuidedFusion:
    """
    ★ AGF — Agreement-Guided Fusion.

    Mechanism
    ---------
    For each image, score how much each base method's attribution map agrees
    with the other two, then set per-image fusion weights from those scores.
    A method that agrees with the consensus is trusted more; a method that
    diverges is down-weighted. Weights are recomputed for every image.

    Mathematical definition
    ------------------------
    Let M_1, M_2, M_3 be the resized, non-negative maps for one image and let
    Sim(.,.) be the agreement kernel (below).

    Per-method agreement (mean similarity with the OTHER K-1 methods):
        a_k = (1 / (K-1)) * sum_{j != k} Sim(M_k, M_j)          (K = 3)

    Weights (softmax at temperature tau, tau = 1 in the reported runs):
        w_k = exp(a_k / tau) / sum_l exp(a_l / tau)

    Fused map (weighted arithmetic mean, then re-normalised to [0, 1]):
        AGF(i,j) = sum_k  w_k * M_k(i,j)

    Agreement kernels
    -----------------
    The kernel controls whether the weights can separate at all, so it is a
    first-class, ablatable choice (`agreement_kernel`):

    "topk_dice"  (DEFAULT, proposed)
        Binarise each map at its OWN top-q quantile (q = `topk_frac`) and take
        the Dice coefficient of the two masks, in [0, 1]. Thresholding is the
        operative step: it discards the shared low-level bulk and compares
        only WHERE each method concentrates its evidence. One parameter (q),
        and q is the same quantity a reader already interprets when they look
        at a thresholded saliency overlay.

    "pearson"
        Sim = mean-centred cosine, i.e. the Pearson correlation between the
        two maps, in [-1, 1]. Parameter-free, and it removes the constant
        offset that inflates raw cosine -- but on a synthetic control where
        all three maps carry the SAME coarse spatial prior and differ only in
        fine structure, it scored a spread of 0.022 against ssim_cosine's
        0.023. Centring removes a DC *offset*, not a shared spatial
        *component*, so it does NOT on its own cure the floor effect. Kept as
        an ablation arm because that negative result is worth reporting.

    "ssim_cosine"  (v6 legacy — the ablation control)
        Sim = 0.5 * [SSIM + raw cosine] on min-max-normalised non-negative
        maps. Both terms are bounded well above zero for any two maps sharing
        a coarse spatial prior; the observed score spread on real data was
        ~0.02, and softmax at tau=1 turned that into a weight spread of
        ~0.007, i.e. the fusion was an arithmetic mean to within 2%.

    On the synthetic shared-prior control (three maps, one fine-structure
    outlier) the agreement-score spreads were:
        ssim_cosine 0.023 | pearson 0.022 | topk_dice 0.110
    All three rank the outlier last; they differ in how much dynamic range
    they give the softmax. Section 18 repeats this on real attributions.

    Note that near-uniform weights are NOT automatically a defect: when the
    three base methods genuinely agree, uniform weights are the correct
    answer, and topk_dice returns them (weight std < 0.01 on the all-agree
    control). What was wrong in v6 is that the kernel returned near-uniform
    weights whether the methods agreed or not. Do not claim agreement-guided
    weighting without reporting the weight-spread diagnostic for the kernel
    actually used; Section 18 is that test.

    Returns
    -------
    fuse() returns a 5-tuple:
        (fused_map, resolved_class_idx, individual_maps, weights, agreement_scores)
    fuse_maps() fuses ALREADY-COMPUTED maps and returns a 4-tuple:
        (fused_map, individual_maps, weights, agreement_scores)
    """

    KERNELS = ("topk_dice", "pearson", "ssim_cosine")

    def __init__(self, methods: Dict[str, "BaseAttributor"],
                 ssim_win_size: int = 7,
                 agreement_kernel: Optional[str] = None,
                 temperature: Optional[float] = None,
                 topk_frac: Optional[float] = None,
                 weight_rule: str = "softmax"):
        agreement_kernel = agreement_kernel or _AGF_DEFAULT_KERNEL
        temperature = _AGF_DEFAULT_TAU if temperature is None else temperature
        topk_frac   = _AGF_DEFAULT_TOPK if topk_frac is None else topk_frac
        if agreement_kernel not in self.KERNELS:
            raise ValueError(f"agreement_kernel must be one of {self.KERNELS}, "
                             f"got {agreement_kernel!r}")
        if weight_rule not in ("softmax", "linear"):
            raise ValueError("weight_rule must be 'softmax' or 'linear'")
        if not (0.0 < float(topk_frac) < 1.0):
            raise ValueError("topk_frac must lie strictly in (0, 1)")
        if float(temperature) <= 0.0:
            raise ValueError("temperature must be > 0")
        self.methods          = methods
        self.ssim_win_size    = ssim_win_size
        self.agreement_kernel = agreement_kernel
        self.temperature      = float(temperature)
        self.topk_frac        = float(topk_frac)
        self.weight_rule      = weight_rule

    # ---- similarity primitives ------------------------------------------

    @staticmethod
    def _cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
        """Raw cosine on the non-negative maps (v6 behaviour)."""
        a_flat = a.flatten().astype(np.float64)
        b_flat = b.flatten().astype(np.float64)
        num = float(np.dot(a_flat, b_flat))
        den = float(np.linalg.norm(a_flat) * np.linalg.norm(b_flat)) + 1e-8
        return num / den

    @staticmethod
    def _pearson_similarity(a: np.ndarray, b: np.ndarray) -> float:
        """Mean-centred cosine == Pearson r, in [-1, 1].

        A constant map (e.g. an identically-zero GradCAM++ on a layer that
        never activates) has zero variance and undefined correlation; it is
        scored 0.0, i.e. 'carries no agreement information', so the softmax
        pushes weight onto the two methods that do agree.
        """
        a_flat = a.flatten().astype(np.float64)
        b_flat = b.flatten().astype(np.float64)
        a_c = a_flat - a_flat.mean()
        b_c = b_flat - b_flat.mean()
        na = float(np.linalg.norm(a_c))
        nb = float(np.linalg.norm(b_c))
        if na < 1e-12 or nb < 1e-12:
            return 0.0
        r = float(np.dot(a_c, b_c) / (na * nb))
        return float(np.clip(r, -1.0, 1.0))

    def _dice_similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        """Dice overlap of the two top-`topk_frac` saliency masks, in [0, 1].

        Each map is thresholded at its OWN quantile, so the two masks always
        have (near-)equal cardinality and Dice is not biased by one method
        being flatter than the other.
        """
        a_flat = a.flatten().astype(np.float64)
        b_flat = b.flatten().astype(np.float64)
        q = 1.0 - self.topk_frac
        ta, tb = np.quantile(a_flat, q), np.quantile(b_flat, q)
        ma, mb = a_flat >= ta, b_flat >= tb
        # A constant map thresholds to the whole image; treat as uninformative.
        if ma.all() or mb.all() or not ma.any() or not mb.any():
            return 0.0
        inter = float(np.logical_and(ma, mb).sum())
        return float(2.0 * inter / (float(ma.sum()) + float(mb.sum()) + 1e-12))

    def _ssim_similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        # a, b are already clipped to >= 0 and roughly in [0, 1] (see fuse()).
        a32, b32 = a.astype(np.float32), b.astype(np.float32)
        win = self.ssim_win_size
        # Guard against tiny maps (win_size must not exceed the smaller
        # spatial dimension) -- shrink to the largest valid odd window.
        max_valid = min(a32.shape) if min(a32.shape) % 2 == 1 else min(a32.shape) - 1
        win = min(win, max_valid) if max_valid >= 3 else 3
        try:
            return float(_skimage_ssim(a32, b32, data_range=1.0, win_size=win))
        except Exception:
            # Last-resort fallback: treat as perfectly-dissimilar rather than
            # crashing the whole fusion on a single degenerate image.
            return 0.0

    def kernel_similarity(self, a: np.ndarray, b: np.ndarray,
                          kernel: Optional[str] = None) -> float:
        """Pairwise agreement under `kernel` (defaults to self.agreement_kernel).

        Public so the Section 18 kernel ablation can score one set of
        already-computed maps under every kernel without re-attributing.
        """
        k = kernel or self.agreement_kernel
        if k == "pearson":
            return self._pearson_similarity(a, b)
        if k == "topk_dice":
            return self._dice_similarity(a, b)
        if k == "ssim_cosine":
            return 0.5 * (self._ssim_similarity(a, b) + self._cosine_similarity(a, b))
        raise ValueError(f"unknown agreement kernel {k!r}")

    def _pairwise_similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        return self.kernel_similarity(a, b, self.agreement_kernel)

    # ---- agreement + weighting -------------------------------------------

    def agreement_scores(self, maps: Dict[str, np.ndarray],
                         kernel: Optional[str] = None) -> Dict[str, float]:
        """a_k = mean similarity of method k with the other K-1 methods."""
        names = list(maps.keys())
        scores: Dict[str, float] = {}
        for name_i in names:
            sims = [self.kernel_similarity(maps[name_i], maps[name_j], kernel)
                    for name_j in names if name_j != name_i]
            scores[name_i] = float(np.mean(sims))
        return scores

    # back-compat alias for any external caller
    _agreement_scores = agreement_scores

    def weights_from_scores(self, scores: Dict[str, float],
                            temperature: Optional[float] = None,
                            rule: Optional[str] = None) -> Dict[str, float]:
        names = list(scores.keys())
        vals  = np.array([scores[n] for n in names], dtype=np.float64)
        rule  = rule or self.weight_rule
        if rule == "linear":
            c = np.clip(vals, 0.0, None)
            tot = c.sum()
            w = (c / tot) if tot > 1e-12 else np.full(len(c), 1.0 / len(c))
        else:
            tau = float(temperature if temperature is not None else self.temperature)
            v = vals / tau
            v = v - v.max()                       # numerical stability
            e = np.exp(v)
            w = e / (e.sum() + 1e-12)
        return {n: float(wi) for n, wi in zip(names, w)}

    def _softmax_weights(self, scores: Dict[str, float]) -> Dict[str, float]:
        return self.weights_from_scores(scores)

    # ---- fusion over PRE-COMPUTED maps (no attributor calls) -------------

    def fuse_maps(self, indiv: Dict[str, np.ndarray]
                  ) -> Tuple[np.ndarray, Dict[str, np.ndarray],
                             Dict[str, float], Dict[str, float]]:
        """Fuse maps that the caller already computed.

        Added in v6 so the evaluation loop can compute each base method
        EXACTLY ONCE. The old path computed GradCAM++/SmoothGrad/IG
        standalone for their own table rows and then fuse() recomputed all
        three internally -- roughly 52 extra forward/backward passes per
        image, about half the total GPU time of a run.

        It also removes a subtler asymmetry: fuse() clipped its constituents
        to >= 0 after resize_map (Lanczos interpolation can undershoot below
        zero even on a [0,1] input) while the standalone arm did not. Gini,
        Sparsity and Entropy for "GradCAM++" and for "the GradCAM++ inside
        AGF" were therefore computed on slightly different arrays. Routing
        both through this method makes them the same array by construction.

        Returns (fused_map, clipped_individual_maps, weights, agreement).
        """
        if not indiv:
            raise ValueError("fuse_maps() received no maps.")
        indiv = {name: np.clip(np.asarray(m, dtype=np.float32), 0.0, None)
                 for name, m in indiv.items()}
        shapes = {m.shape for m in indiv.values()}
        if len(shapes) != 1:
            raise ValueError(f"fuse_maps() got mismatched map shapes: {shapes}. "
                             "Resize every map to target_size first.")

        agreement = self.agreement_scores(indiv)
        weights   = self.weights_from_scores(agreement)

        fused = np.zeros_like(next(iter(indiv.values())), dtype=np.float64)
        for name, w in weights.items():
            fused += w * indiv[name].astype(np.float64)

        fused = np.clip(fused, 0.0, None)
        fused = fused / (fused.max() + 1e-8)
        fused = np.nan_to_num(fused, nan=0.0, posinf=0.0, neginf=0.0)
        return fused.astype(np.float32), indiv, weights, agreement

    # ---- main entry point --------------------------------------------------

    def fuse(self, tensor, class_idx: Optional[int] = None,
              target_size: Optional[Tuple[int, int]] = None
              ) -> Tuple[np.ndarray, int, Dict[str, np.ndarray],
                         Dict[str, float], Dict[str, float]]:
        """
        Returns (fused_map, resolved_class_idx, individual_maps,
                 weights, agreement_scores).
        """
        indiv: Dict[str, np.ndarray] = {}
        resolved_class = class_idx
        for name, method in self.methods.items():
            attr, pred = method(tensor, resolved_class)
            if resolved_class is None:
                resolved_class = pred
            if target_size is not None:
                attr = resize_map(attr, target_size)
            indiv[name] = attr
        fused, indiv, weights, agreement = self.fuse_maps(indiv)
        return fused, resolved_class, indiv, weights, agreement

    def freeze_noise(self, enable: bool = True) -> "AgreementGuidedFusion":
        """Forward noise freezing to any stochastic constituent.

        Required so SaliencyMetrics.sensitivity() can hold SmoothGrad's noise
        draw fixed across the (base, perturbed) pair when measuring the FUSED
        method, exactly as it does when measuring SmoothGrad standalone.
        Without this the AGF row of the robustness table carries Monte-Carlo
        variance that the GradCAM++ and IG rows do not.
        """
        for m in self.methods.values():
            if hasattr(m, "freeze_noise"):
                m.freeze_noise(enable)
        return self


# Short alias used in the narrative/figures.
AGF = AgreementGuidedFusion
print("xai_attribution final-1.0 (v7.1 numerics) loaded — methods: GradCAM++, SmoothGrad, IG; "
      "proposed fusion: AGF (AgreementGuidedFusion). "
      "kernel=%s; _normalize fails loudly on degenerate maps." % _AGF_DEFAULT_KERNEL)

Writing /kaggle/working/xai_attribution.py


In [11]:
# ── §8–9  Import the freshly-written module ─────────────────────────────
import importlib
sys.path.insert(0, "/kaggle/working")
import xai_attribution as _xai
importlib.reload(_xai)
# The two retired fusion variants are no longer importable: both classes were
# deleted from xai_attribution.py in v6.
from xai_attribution import (
    BaseAttributor, resize_map, _safe_entropy,
    GradCAMPlusPlus, SmoothGrad, IntegratedGradients,
    AgreementGuidedFusion, SaliencyMetrics,
)
print("Imported:", [c.__name__ for c in
      (GradCAMPlusPlus, SmoothGrad, IntegratedGradients,
       AgreementGuidedFusion, SaliencyMetrics)])
# Hard guard: if a stale xai_attribution.py is on the path (e.g. from an
# earlier session's /kaggle/working), the fixes below will silently not apply.
assert hasattr(AgreementGuidedFusion, "fuse_maps"), \
    "Stale xai_attribution.py imported — re-run the %%writefile cell above."
assert hasattr(SmoothGrad, "freeze_noise"), \
    "Stale xai_attribution.py imported — re-run the %%writefile cell above."
print("v7 module verified: fuse_maps + freeze_noise present.")

# ---- push the CONFIG kernel choice into the freshly-written module --------
#   The module is emitted by %%writefile and therefore cannot see notebook
#   globals at definition time. Setting the three module-level defaults here
#   means AGREEMENT_KERNEL in Section 2 is the ONE place the kernel is chosen;
#   every AgreementGuidedFusion(...) construction site inherits it.
#   `_xai` is the module object bound by the import above.
_xai._AGF_DEFAULT_KERNEL = AGREEMENT_KERNEL
_xai._AGF_DEFAULT_TAU    = AGREEMENT_TAU
_xai._AGF_DEFAULT_TOPK   = TOPK_FRAC
assert AgreementGuidedFusion({}).agreement_kernel == AGREEMENT_KERNEL, \
    "kernel override did not take effect"
assert AgreementGuidedFusion({}).temperature == AGREEMENT_TAU
for _k in KERNEL_ABLATION:
    assert _k in AgreementGuidedFusion.KERNELS, f"unknown kernel in KERNEL_ABLATION: {_k}"
assert AgreementGuidedFusion({}).topk_frac == TOPK_FRAC
assert AgreementGuidedFusion({}).weight_rule == AGF_WEIGHT_RULE
XAI_MODULE_SHA256 = hashlib.sha256(open(_xai.__file__, "rb").read()).hexdigest()
print(f"AGF agreement kernel = {AGREEMENT_KERNEL} (tau={AGREEMENT_TAU}, topk={TOPK_FRAC})")
print(f"xai_attribution.py sha256 = {XAI_MODULE_SHA256}")

xai_attribution final-1.0 (v7.1 numerics) loaded — methods: GradCAM++, SmoothGrad, IG; proposed fusion: AGF (AgreementGuidedFusion). kernel=topk_dice; _normalize fails loudly on degenerate maps.
xai_attribution final-1.0 (v7.1 numerics) loaded — methods: GradCAM++, SmoothGrad, IG; proposed fusion: AGF (AgreementGuidedFusion). kernel=topk_dice; _normalize fails loudly on degenerate maps.
Imported: ['GradCAMPlusPlus', 'SmoothGrad', 'IntegratedGradients', 'AgreementGuidedFusion', 'SaliencyMetrics']
v7 module verified: fuse_maps + freeze_noise present.
AGF agreement kernel = topk_dice (tau=1.0, topk=0.1)
xai_attribution.py sha256 = aa778d17494935c864236f8803edf7be9830d1549d08a2a43f6f9df32803b0e2


In [12]:
# ── §10  Sanity check: agreement-kernel self-check (synthetic, no GPU) ────
#   Runs BEFORE any attribution work. A kernel that cannot separate agreeing
#   from disagreeing maps silently degrades AGF into an arithmetic mean, and
#   the only symptom is a near-uniform weight table 12 hours later. This cell
#   fails loudly instead.
#
#   Three synthetic maps share a coarse spatial prior (as real attribution
#   maps do) and differ only in fine structure. In the DISAGREE case one map
#   puts its fine structure elsewhere; in the AGREE case all three match.
def _kernel_selfcheck(verbose=True):
    H = W = IMG_SIZE
    yy, xx = np.mgrid[0:H, 0:W]

    def _blob(cy, cx, s):
        m = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2.0 * s * s))
        return (m / m.max()).astype(np.float32)

    _common = _blob(H // 2, W // 2, H // 4)

    def _mm(cy, cx, s, seed):
        r = np.random.default_rng(seed)
        m = 0.72 * _common + 0.28 * _blob(cy, cx, s)
        m = np.clip(m + 0.03 * r.standard_normal((H, W)), 0, None)
        return (m / m.max()).astype(np.float32)

    disagree = {"gradcam": _mm(95, 95, 20, 1), "smoothgrad": _mm(99, 98, 22, 2),
                "ig": _mm(150, 155, 20, 3)}
    agree    = {"gradcam": _mm(95, 95, 20, 11), "smoothgrad": _mm(97, 96, 21, 12),
                "ig": _mm(96, 98, 20, 13)}

    rows = []
    for k in AgreementGuidedFusion.KERNELS:
        f = AgreementGuidedFusion({}, agreement_kernel=k)
        sd = f.agreement_scores(disagree)
        wd = np.array(list(f.weights_from_scores(sd).values()))
        wa = np.array(list(f.weights_from_scores(f.agreement_scores(agree)).values()))
        sv = np.array(list(sd.values()))
        rows.append({"kernel": k, "score_spread": float(sv.max() - sv.min()),
                     "w_std_disagree": float(wd.std()), "w_std_agree": float(wa.std()),
                     "ratio": float(wd.std() / (wa.std() + 1e-12)),
                     "outlier_last": bool(min(sd, key=sd.get) == "ig")})
    df = pd.DataFrame(rows)
    if verbose:
        print("AGREEMENT-KERNEL SELF-CHECK (synthetic shared-prior control)")
        print(df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

    live = df[df.kernel == AGREEMENT_KERNEL].iloc[0]
    assert live["outlier_last"], \
        f"{AGREEMENT_KERNEL}: failed to rank the outlier method last"
    assert live["ratio"] > 3.0, (
        f"{AGREEMENT_KERNEL}: weight spread barely responds to disagreement "
        f"(ratio={live['ratio']:.2f}). The fusion is an arithmetic mean in "
        f"disguise -- pick a different AGREEMENT_KERNEL before spending GPU time.")
    # The ratio alone is not enough: a kernel can respond proportionally while
    # its ABSOLUTE separation stays negligible (pearson does exactly this --
    # ratio ~99x, but weights of 0.336/0.336/0.329). Require real separation.
    assert live["w_std_disagree"] > 0.01, (
        f"{AGREEMENT_KERNEL}: absolute weight separation under clear "
        f"disagreement is only {live['w_std_disagree']:.4f} (uniform = 0.0000, "
        f"usable > 0.01). The weighting cannot meaningfully re-rank the base "
        f"methods; do not claim agreement-guided weighting with this kernel.")

    # NaN safety on a degenerate all-zero map (the VGG16-BN GradCAM++ case)
    deg = dict(disagree); deg["gradcam"] = np.zeros((H, W), np.float32)
    for k in AgreementGuidedFusion.KERNELS:
        fu, _, w, _ = AgreementGuidedFusion({}, agreement_kernel=k).fuse_maps(deg)
        assert np.isfinite(fu).all(), f"{k}: non-finite fused map on a constant input"
        assert abs(sum(w.values()) - 1.0) < 1e-9, f"{k}: weights do not sum to 1"

    if verbose:
        print(f"\n[OK] {AGREEMENT_KERNEL}: outlier down-weighted; weight spread "
              f"{live['ratio']:.1f}x larger under disagreement than agreement; "
              f"degenerate maps handled without NaN.")
    return df

_KERNEL_SELFCHECK = _kernel_selfcheck()
# weight-simplex check on random non-negative maps, every kernel, several taus
_rng = np.random.default_rng(SEED)
for _k in AgreementGuidedFusion.KERNELS:
    for _tau in [AGREEMENT_TAU] + TEMP_GRID:
        _m = {n: _rng.random((32, 32)).astype(np.float32) for n in ("gradcam", "smoothgrad", "ig")}
        _f = AgreementGuidedFusion({}, agreement_kernel=_k, temperature=_tau)
        _fu, _, _w, _ = _f.fuse_maps(_m)
        assert abs(sum(_w.values()) - 1.0) < 1e-9 and min(_w.values()) > 0.0
        assert np.isfinite(_fu).all() and _fu.max() <= 1.0 + 1e-6 and _fu.min() >= 0.0
print("[OK] weights on the simplex, fused maps finite and in [0, 1] for all kernels/taus.")
_KERNEL_SELFCHECK_PATH = os.path.join(DIRS["Statistics"], "agf_kernel_selfcheck.csv")
_KERNEL_SELFCHECK.to_csv(_KERNEL_SELFCHECK_PATH, index=False)   # always persisted
mark_section("10_sanity_checks")

AGREEMENT-KERNEL SELF-CHECK (synthetic shared-prior control)
     kernel  score_spread  w_std_disagree  w_std_agree   ratio  outlier_last
  topk_dice        0.1126          0.0171       0.0002 75.6501          True
    pearson        0.0220          0.0034       0.0000 99.3128          True
ssim_cosine        0.0219          0.0034       0.0005  7.2402          True

[OK] topk_dice: outlier down-weighted; weight spread 75.7x larger under disagreement than agreement; degenerate maps handled without NaN.
[OK] weights on the simplex, fused maps finite and in [0, 1] for all kernels/taus.


In [13]:
# ── §11a  Reusable machinery (defined once, reused per backbone) ──────────

# ---- AOPC insertion/deletion curve (grey baseline, softmax) ----------------
_NORM_MEAN_T = torch.tensor(NORM_MEAN, dtype=torch.float32).view(1, 3, 1, 1)
_NORM_STD_T  = torch.tensor(NORM_STD,  dtype=torch.float32).view(1, 3, 1, 1)

def neutral_baseline(inp: torch.Tensor, pixel_value: float = AOPC_BASELINE_PIXEL):
    """Constant-pixel baseline expressed in NORMALISED space.

    The inputs reaching this function have already been through
    Normalize(mean=NORM_MEAN, std=NORM_STD), so a scalar fill is not a
    neutral image: torch.full_like(inp, 0.5) back-projects to RGB
    (0.600, 0.568, 0.519), a warm tan. Neutral grey has to be computed
    per channel as (pixel - mean) / std -> (0.066, 0.196, 0.418).
    """
    m = _NORM_MEAN_T.to(inp.device, inp.dtype)
    s = _NORM_STD_T.to(inp.device, inp.dtype)
    return ((pixel_value - m) / s).expand_as(inp).contiguous()

def rank_pixels_desc(flat, seed: int = AOPC_TIE_BREAK_SEED):
    """Rank flattened attribution values descending, with SEEDED RANDOM
    tie-breaking.

    WHY THE PREVIOUS VERSION WAS WRONG
    ----------------------------------
    It used `np.argsort(flat)[::-1]`. np.argsort is stable-ascending, so
    reversing it returns tied values in DESCENDING INDEX order -- a
    deterministic bottom-right-to-top-left raster scan. Consequences:

      * any map with large flat regions is scored partly on image geometry
        rather than on attribution quality;
      * a fully constant map is scored ENTIRELY on geometry. The reported
        VGG16-BN GradCAM++ AOPC-Ins of 0.6678 was the score of revealing
        pixels from the bottom-right corner inward. It measured nothing
        about GradCAM++ or about the model.

    lexsort with a seeded permutation as the secondary key removes the
    spatial prior while keeping the run reproducible. NOTE: np.lexsort takes
    the LAST key as primary, so `-flat` drives the ordering and the
    permutation only resolves exact ties.

    The seed is a fixed CONSTANT, not per-image and not per-method: the tie
    order must be identical across methods so the paired Wilcoxon compares
    like with like. A per-call seed would inject variance into the pairing
    rather than remove bias from it.
    """
    flat = np.asarray(flat, dtype=np.float64).ravel()
    rng = np.random.default_rng(seed)
    tiebreak = rng.permutation(flat.size)
    return np.lexsort((tiebreak, -flat))

def aopc_curve(model, tensor, attr_map, class_idx,
               n_steps=AOPC_N_STEPS, patch_pct=AOPC_PATCH_PCT, mode="insert"):
    """mode='insert': reveal top-k pixels from grey -> higher = better.
       mode='delete': mask top-k pixels with grey    -> lower  = better.
       Returns a list of n_steps softmax-confidence scores.

       Both modes share ONE ranking (most-important-first) so insertion and
       deletion are exact complements of each other.
    """
    inp    = tensor.unsqueeze(0).to(DEVICE)
    flat   = attr_map.flatten()
    n_pix  = len(flat)
    ranked = np.ascontiguousarray(rank_pixels_desc(flat))
    grey   = neutral_baseline(inp)
    scores = []
    perturbed = mask = None
    with torch.inference_mode():
        for step in range(1, n_steps + 1):
            k = int(step * patch_pct * n_pix)
            mask = torch.zeros(n_pix, dtype=torch.float32, device=DEVICE)
            mask[ranked[:k]] = 1.0
            mask = mask.reshape(attr_map.shape).unsqueeze(0).unsqueeze(0)
            perturbed = (grey * (1 - mask) + inp * mask) if mode == "insert" \
                        else (inp * (1 - mask) + grey * mask)
            s = float(torch.softmax(model(perturbed), dim=1)[0, class_idx].item())
            scores.append(s)
    del inp, grey, mask, perturbed
    return scores


# ---- per-map metrics (numpy, cheap; map assumed >=0, roughly [0,1]) --------
def map_entropy(m: np.ndarray) -> float:
    return _safe_entropy(np.clip(m, 0, None))

def map_sparsity(m: np.ndarray, top_frac: float = 0.10) -> float:
    """Fraction of total attribution mass concentrated in the top `top_frac`
    of pixels. Higher = sparser / more focused. Complements Gini."""
    a = np.abs(m).flatten().astype(np.float64)
    tot = a.sum()
    if tot < 1e-8:
        return 0.0
    k = max(1, int(top_frac * a.size))
    topk = np.sort(a)[-k:].sum()
    return float(topk / tot)

map_gini = SaliencyMetrics.gini


# ---- per-image scalar accumulator (NO arrays kept in lists) ---------------
@dataclass
class MethodAccumulator:
    ins_aucs:    List[float]       = field(default_factory=list)
    del_aucs:    List[float]       = field(default_factory=list)
    ins_curves:  List[List[float]] = field(default_factory=list)
    del_curves:  List[List[float]] = field(default_factory=list)
    gini:        List[float]       = field(default_factory=list)
    sparsity:    List[float]       = field(default_factory=list)
    entropy:     List[float]       = field(default_factory=list)
    runtime_ms:  List[float]       = field(default_factory=list)
    sens:        List[float]       = field(default_factory=list)   # robustness
    sens_idxs:   List[int]         = field(default_factory=list)   # image idx per sens value
    sample_idxs: List[int]         = field(default_factory=list)
    preds:       List[int]         = field(default_factory=list)
    labels:      List[int]         = field(default_factory=list)
    n_ok: int = 0


# ---- statistics: bootstrap CI + effect sizes ------------------------------
def bootstrap_ci(scores, n_boot=BOOTSTRAP_N, alpha=0.05, seed=SEED,
                 stat=np.mean):
    """Percentile bootstrap CI for `stat`.

    `stat` MUST match the point estimate you report alongside the interval.
    The previous version always resampled the mean while paired_compare
    reported a median, which is why the IG row's CI could sit entirely to
    one side of its own median_diff. Pass stat=np.median for paired
    differences, stat=np.mean for per-method score means.
    """
    rng = np.random.default_rng(seed)
    arr = np.asarray(scores, dtype=np.float64)
    if arr.size == 0:
        return (float("nan"), float("nan"))
    idx  = rng.integers(0, arr.size, size=(n_boot, arr.size))
    boot = stat(arr[idx], axis=1)
    lo, hi = np.percentile(boot, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    del boot, idx
    return float(lo), float(hi)

def cliffs_delta(a, b) -> float:
    """Cliff's delta in [-1,1]; >0 means a tends to exceed b. O(n log n)."""
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    na, nb = a.size, b.size
    if na == 0 or nb == 0:
        return 0.0
    b_sorted = np.sort(b)
    greater = np.searchsorted(b_sorted, a, side="left").sum()      # #(a>b)
    less    = (nb * na) - np.searchsorted(b_sorted, a, side="right").sum()  # #(a<b)
    return float((greater - less) / (na * nb))

def paired_rank_biserial(a, b) -> float:
    """Matched-pairs rank-biserial correlation for the Wilcoxon signed-rank
    test (Kerby 2014): r = (sum ranks favouring a - favouring b) / total.
    Range [-1,1]; +1 means a>b on every non-zero pair."""
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    d = a - b
    d = d[d != 0]
    if d.size == 0:
        return 0.0
    ranks = pd.Series(np.abs(d)).rank().to_numpy()
    r_pos = ranks[d > 0].sum()
    r_neg = ranks[d < 0].sum()
    tot = r_pos + r_neg
    return float((r_pos - r_neg) / tot) if tot > 0 else 0.0

def cliff_magnitude(delta: float) -> str:
    """Romano et al. (2006) thresholds -- valid for Cliff's delta ONLY."""
    d = abs(delta)
    return ("negligible" if d < 0.147 else "small" if d < 0.33
            else "medium" if d < 0.474 else "large")

def rb_magnitude(r: float) -> str:
    """Cohen's conventions for correlation-family effect sizes, applied to
    the matched-pairs rank-biserial r. State these thresholds in the paper.

    This is the effect size that belongs to the Wilcoxon signed-rank test.
    Cliff's delta is an UNPAIRED dominance statistic; it ignores the pairing
    the test relies on and can disagree sharply (e.g. rank-biserial -0.337,
    'moderate', vs delta -0.084, 'negligible', on the same comparison).
    """
    r = abs(r)
    return ("negligible" if r < 0.10 else "small" if r < 0.30
            else "medium" if r < 0.50 else "large")


# ---- inferential statistics ------------------------------------------------
# FINAL: the per-backbone `paired_compare()` of v7.1 was REMOVED. It tested only
# insertion and robustness (never deletion), never compared against Equal Fusion,
# applied Holm within a 3-test family per backbone+metric, and reported a paired
# MEDIAN next to a Wilcoxon p (which can disagree in sign on skewed data -- see the
# ResNet50 v7.1 row). All inference now happens ONCE, in §15, from the per-image
# CSVs, with pre-declared families (CONFIG: PRIMARY_* / SECONDARY_*).

# ---- map post-processing: ONE definition used by every loop ---------------
def postprocess_map(m: np.ndarray) -> np.ndarray:
    """CONFIG MAP_POSTPROC: Lanczos resize to IMG_SIZE x IMG_SIZE, clip >= 0.
    Identical to what fuse_maps() does to its constituents, so the standalone
    rows, the fusion inputs and the robustness maps are the same arrays."""
    return np.clip(resize_map(np.asarray(m, dtype=np.float32), (IMG_SIZE, IMG_SIZE)),
                   0.0, None).astype(np.float32)

EQUAL_WEIGHTS = {"gradcam": 1.0 / 3, "smoothgrad": 1.0 / 3, "ig": 1.0 / 3}

# ---- attributor factory (fresh per image; hooks never accumulate) ---------
def make_base_generators(model, target_layer):
    gc_gen = GradCAMPlusPlus(model, target_layer)
    sg_gen = SmoothGrad(model, n_samples=SG_N_SAMPLES, noise_std=SG_NOISE_STD,
                        seed=SEED, noise_mode=SG_NOISE_MODE,
                        noise_frac=SG_NOISE_FRAC, seed_mode=SG_SEED_MODE)
    ig_gen = IntegratedGradients(model, n_steps=IG_N_STEPS)
    return gc_gen, sg_gen, ig_gen

def fusion_dict(gc_gen, sg_gen, ig_gen):
    return {"gradcam": gc_gen, "smoothgrad": sg_gen, "ig": ig_gen}


class _MethodCallable:
    """Callable attributor wrapper that also forwards freeze_noise().

    SaliencyMetrics.sensitivity() freezes any stochastic estimator across
    the (base, perturbed) pair so the metric measures input sensitivity and
    not Monte-Carlo variance. That only works if the object it is handed
    exposes freeze_noise(). make_method_callable used to return bare
    closures, so the freeze silently did nothing for SmoothGrad and AGF --
    the two stochastic rows -- while GradCAM++ and IG were deterministic.
    The robustness column was therefore not measuring the same quantity
    across rows.
    """
    __slots__ = ("_fn", "_stochastic", "_cleanup")

    def __init__(self, fn, stochastic=(), cleanup=None):
        self._fn = fn
        self._stochastic = list(stochastic)
        self._cleanup = cleanup

    def __call__(self, t, c=None):
        return self._fn(t, c)

    def freeze_noise(self, enable: bool = True):
        for g in self._stochastic:
            if hasattr(g, "freeze_noise"):
                g.freeze_noise(enable)
        return self

    def close(self):
        if self._cleanup is not None:
            self._cleanup()


def make_method_callable(key, model, target_layer):
    """Return a fresh attributor callable f(tensor, class_idx)->(map, pred)
    for robustness/sensitivity.

    GAF and a reliability-weighted variant branches removed: both classes are gone from
    xai_attribution.py (retired from the experiment, ~250 lines of
    unreachable code). Re-adding them means restoring the classes first.

    Generators are now built ONCE per callable rather than per call, so a
    single sensitivity() evaluation reuses one SmoothGrad instance and its
    frozen noise cache survives across the base and perturbed calls. The
    GradCAM++ branch still rebuilds per call because its hooks must not
    accumulate; it is deterministic, so nothing is frozen there.
    """
    # FINAL (REQUIRES RERUN of robustness): every callable returns the
    # POST-PROCESSED 224x224 map (postprocess_map). v7.1 returned GradCAM++ at its
    # native 7x7 resolution here while IG/SmoothGrad/AGF were 224x224 and the
    # faithfulness loop used the resized maps -- the robustness column therefore
    # measured different objects for different rows. For IG and SmoothGrad the
    # post-processing is a no-op (already 224x224 and >= 0), so their values are
    # unchanged; GradCAM++ changes; Equal Fusion is new.
    if key == "gradcam_pp":
        def _fn(t, c=None):
            gc = GradCAMPlusPlus(model, target_layer)
            try:
                m, p = gc.generate(t, c)
                return postprocess_map(m), p
            finally: gc.remove()
        return _MethodCallable(_fn)

    if key == "ig":
        ig = IntegratedGradients(model, n_steps=IG_N_STEPS)
        def _fn(t, c=None):
            m, p = ig.generate(t, c)
            return postprocess_map(m), p
        return _MethodCallable(_fn)

    if key == "smoothgrad":
        sg = SmoothGrad(model, n_samples=SG_N_SAMPLES, noise_std=SG_NOISE_STD,
                        seed=SEED, noise_mode=SG_NOISE_MODE,
                        noise_frac=SG_NOISE_FRAC, seed_mode=SG_SEED_MODE)
        def _fn(t, c=None):
            m, p = sg.generate(t, c)
            return postprocess_map(m), p
        return _MethodCallable(_fn, stochastic=[sg])

    if key in ("agf", "equal"):
        gc, sg, ig = make_base_generators(model, target_layer)
        fuser = AgreementGuidedFusion(fusion_dict(gc, sg, ig))
        def _fn(t, c=None, _key=key):
            m, p, indiv, _, _ = fuser.fuse(t, class_idx=c,
                                           target_size=(IMG_SIZE, IMG_SIZE))
            if _key == "equal":
                m = weighted_arith(indiv, EQUAL_WEIGHTS)
            return m, p
        return _MethodCallable(_fn, stochastic=[sg], cleanup=gc.remove)

    raise KeyError(f"Unknown method key {key!r}. Valid keys: {ORDER}")


# ---- AGF ablation fusers (ONE fuse per image, no recomputation) ------------
def _renorm(m):
    m = np.clip(m, 0.0, None)
    return (m / (m.max() + 1e-8)).astype(np.float32)

def weighted_arith(indiv: Dict[str, np.ndarray], weights: Dict[str, float]) -> np.ndarray:
    fused = np.zeros_like(next(iter(indiv.values())), dtype=np.float64)
    for name, w in weights.items():
        fused += w * indiv[name].astype(np.float64)
    return _renorm(fused)

def agf_ablation_maps(gc_gen, sg_gen, ig_gen, tensor, pred, fixed_weights=None):
    """One AGF call -> the three ablation variants:
       agf_full : per-image agreement-weighted softmax weights (PROPOSED)
       agf_equal: uniform weights = 1/K   (agreement weighting REMOVED)
       agf_fixed: static weights calibrated on the VALIDATION split (§11b);
                  if None, falls back to equal (logged by the caller)
    Returns (maps_dict, agreement_weights_dict).

    Signature and return type are unchanged so Section 16 keeps working.
    """
    fused_full, _, indiv, w_agree, _ = AgreementGuidedFusion(
        fusion_dict(gc_gen, sg_gen, ig_gen)
    ).fuse(tensor, class_idx=pred, target_size=(IMG_SIZE, IMG_SIZE))
    names = list(indiv.keys())
    eq = {n: 1.0 / len(names) for n in names}
    if fixed_weights is None:
        fixed_weights = eq
    fx = {n: float(fixed_weights.get(n, eq[n])) for n in names}
    s = sum(fx.values()) or 1.0
    fx = {n: v / s for n, v in fx.items()}
    maps = {
        "agf_full":  fused_full,
        "agf_equal": weighted_arith(indiv, eq),
        "agf_fixed": weighted_arith(indiv, fx),
    }
    return maps, w_agree

print("Reusable machinery ready.")

Reusable machinery ready.


# Figures

In [14]:
# ── R1 · Shared helpers ─────────────────────────────────────────────────────
from scipy.stats import spearmanr, pearsonr, binomtest, mannwhitneyu

_W_ENGINE = AgreementGuidedFusion({})          # weights only; kernel/topk from CONFIG

def base_maps(eval_model, target_layer, tensor, class_idx=None, sg_gen=None, ig_gen=None):
    """GradCAM++/SmoothGrad/IG once -> (indiv maps, agreement scores, pred).
    Identical path to the localisation notebook (generate -> resize_map -> fuse_maps)."""
    gc_gen = GradCAMPlusPlus(eval_model, target_layer)
    if sg_gen is None:
        sg_gen = SmoothGrad(eval_model, n_samples=SG_N_SAMPLES, noise_std=SG_NOISE_STD, seed=SEED,
                            noise_mode=SG_NOISE_MODE, noise_frac=SG_NOISE_FRAC, seed_mode=SG_SEED_MODE)
    if ig_gen is None:
        ig_gen = IntegratedGradients(eval_model, n_steps=IG_N_STEPS)
    try:
        with torch.inference_mode(False):
            gc_map, pred = gc_gen.generate(tensor, class_idx)
            sg_map, _    = sg_gen.generate(tensor, class_idx=pred)
            ig_map, _    = ig_gen.generate(tensor, class_idx=pred)
    finally:
        gc_gen.remove()
    pre = {"gradcam":    resize_map(gc_map, (IMG_SIZE, IMG_SIZE)),
           "smoothgrad": resize_map(sg_map, (IMG_SIZE, IMG_SIZE)),
           "ig":         resize_map(ig_map, (IMG_SIZE, IMG_SIZE))}
    fused1, indiv, w1, scores = _W_ENGINE.fuse_maps(pre)
    return indiv, scores, int(pred), fused1, w1

def agf_at(indiv, scores, tau):
    """AGF fused map + weights at temperature tau (scores do not depend on tau)."""
    w = _W_ENGINE.weights_from_scores(scores, temperature=tau)
    return weighted_arith(indiv, w), w

def method_maps(indiv, scores, taus):
    """All evaluated maps for one image. taus: {name: tau}."""
    out = {"gradcam_pp": indiv["gradcam"], "ig": indiv["ig"], "smoothgrad": indiv["smoothgrad"],
           "equal": weighted_arith(indiv, EQUAL_WEIGHTS)}
    ws = {}
    for name, tau in taus.items():
        out[name], ws[name] = agf_at(indiv, scores, tau)
    return out, ws

def aopc_pair(eval_model, tensor, amap, c):
    ins = float(np.mean(aopc_curve(eval_model, tensor, amap, c, mode="insert")))
    dele = float(np.mean(aopc_curve(eval_model, tensor, amap, c, mode="delete")))
    return ins, 1.0 - dele

@torch.inference_mode()
def clean_confidence(eval_model, tensor, c):
    p = torch.softmax(eval_model(tensor.unsqueeze(0).to(DEVICE)).float(), dim=1)[0]
    return float(p[c].item())

def wstats(w):
    v = np.array([w["gradcam"], w["smoothgrad"], w["ig"]], dtype=np.float64)
    return float(v.std()), float(np.abs(v - 1.0 / 3).sum())

def raw_fusion_gap(indiv, w):
    """max_p |sum_m (w_m - 1/3) M_m(p)| BEFORE rescaling; bounded by sum_m |w_m - 1/3|."""
    d = np.zeros_like(indiv["gradcam"], dtype=np.float64)
    for n in ("gradcam", "smoothgrad", "ig"):
        d += (w[n] - 1.0 / 3) * indiv[n].astype(np.float64)
    return float(np.abs(d).max())

def iter_backbones():
    for spec in MODELS:
        if spec["key"] not in RUN_MODELS:
            continue
        model, eval_model, _, backbone_name, _ = load_model(spec)
        target_layer = get_target_layer(model, backbone_name)
        try:
            yield spec, model, eval_model, target_layer
        finally:
            del model, eval_model, target_layer
            free_gpu()

# one-off equivalence check: weighted_arith(...) at tau=1 == fuse_maps() fused map
_chk = {"gradcam": np.random.default_rng(0).random((IMG_SIZE, IMG_SIZE)).astype(np.float32),
        "smoothgrad": np.random.default_rng(1).random((IMG_SIZE, IMG_SIZE)).astype(np.float32),
        "ig": np.random.default_rng(2).random((IMG_SIZE, IMG_SIZE)).astype(np.float32)}
_f, _ind, _w, _s = _W_ENGINE.fuse_maps(_chk)
_f2, _ = agf_at(_ind, _s, AGREEMENT_TAU)
assert np.max(np.abs(_f - _f2)) < 1e-6, "agf_at() does not reproduce fuse_maps()"
print("R1 helpers ready; agf_at() reproduces fuse_maps() at tau=1.")

R1 helpers ready; agf_at() reproduces fuse_maps() at tau=1.


In [15]:
# ── R19 · Insertion/deletion curves, redrawn from the saved per-step CSVs ─────
#   CPU only. Needs the agreement-guided-fusion-agf-final OUTPUT attached as Input
#   (it contains <backbone>_aopc_curves.csv). Saves one PDF per backbone:
#   Figures/aopc_curves_<backbone>.pdf. No built-in title; consistent labels.
#   Lines: mean over images; bands: 95% confidence interval of the mean.
import glob, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

files = sorted(glob.glob("/kaggle/input/**/*_aopc_curves.csv", recursive=True))
assert files, "No *_aopc_curves.csv found: attach the agf-final notebook output as Input."
OUT_DIR = "/kaggle/working/Figures"; os.makedirs(OUT_DIR, exist_ok=True)

LABEL = {"gradcam_pp": "Grad-CAM++", "ig": "IG", "smoothgrad": "SmoothGrad",
         "equal": "Equal Fusion", "agf": "AGF (τ = 1)"}
COLOR = {"gradcam_pp": "#d62728", "ig": "#2ca02c", "smoothgrad": "#1f77b4",
         "equal": "#7f7f7f", "agf": "#ff7f0e"}
plt.rcParams.update({"font.size": 8, "pdf.fonttype": 42})

for path in files:
    key = os.path.basename(path).replace("_aopc_curves.csv", "")
    df = pd.read_csv(path)
    print(f"Using: {path}  ({df.img_idx.nunique()} images)")
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9), sharey=True)
    for ax, mode, title in [(axes[0], "insert", "Insertion"), (axes[1], "delete", "Deletion")]:
        for mk in ["gradcam_pp", "ig", "smoothgrad", "equal", "agf"]:
            sub = df[(df["mode"] == mode) & (df["method_key"] == mk)]
            if sub.empty:
                continue
            g = sub.groupby("frac_pixels")["score"]
            m, se, n = g.mean(), g.std(ddof=1), g.count()
            ci = 1.96 * se / np.sqrt(n)
            x = m.index.values * 100
            ax.plot(x, m.values, color=COLOR[mk], lw=1.4, marker="o", ms=2.5, label=LABEL[mk])
            ax.fill_between(x, (m - ci).values, (m + ci).values, color=COLOR[mk], alpha=0.15, lw=0)
        ax.set_title(title, fontsize=9, fontweight="bold")
        ax.set_xlabel("Pixels revealed (%)" if mode == "insert" else "Pixels removed (%)")
        ax.set_xticks([20, 40, 60, 80, 100]); ax.grid(alpha=0.3, lw=0.5)
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
    axes[0].set_ylabel("Target-class probability")
    axes[1].legend(fontsize=7, frameon=False, loc="upper right")
    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(OUT_DIR, f"aopc_curves_{key}.{ext}"), dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", os.path.join(OUT_DIR, f"aopc_curves_{key}.pdf"))


Using: /kaggle/input/notebooks/tamannajarintima/agreement-guided-fusion-agf-final/Statistics/per_image/densenet121_aopc_curves.csv  (200 images)
Saved: /kaggle/working/Figures/aopc_curves_densenet121.pdf
Using: /kaggle/input/notebooks/tamannajarintima/agreement-guided-fusion-agf-final/Statistics/per_image/resnet50_aopc_curves.csv  (200 images)
Saved: /kaggle/working/Figures/aopc_curves_resnet50.pdf
Using: /kaggle/input/notebooks/tamannajarintima/agreement-guided-fusion-agf-final/Statistics/per_image/vgg16_aopc_curves.csv  (200 images)
Saved: /kaggle/working/Figures/aopc_curves_vgg16.pdf


In [16]:
# ── R20 · Qualitative grid, regenerated (GPU) ───────────────────────────────
#   Uses the setup cells and R1 above. Same image selection as the original grid
#   (first correctly classified test image of each class), same backbone.
#   Columns: MRI, Grad-CAM++, IG, SmoothGrad, Equal Fusion, AGF (tau = 1), AGF (tau*).
#   No built-in title; consistent labels. Saves Figures/qualitative_grid.pdf.
import json, glob

BACKBONE = "densenet121"
# tau* frozen in R2 of the original revision run (validation-selected)
_tau_files = glob.glob("/kaggle/input/**/tau_selection.json", recursive=True) + \
             glob.glob("/kaggle/working/**/tau_selection.json", recursive=True)
TAU_STAR = (json.load(open(_tau_files[0]))["tau_selected"][BACKBONE]
            if _tau_files else 0.25)            # 0.25 = value frozen for DenseNet121
print("tau* =", TAU_STAR, "from", _tau_files[0] if _tau_files else "default (frozen value)")

ROW_LABEL = {"glioma": "Glioma", "meningioma": "Meningioma",
             "notumor": "No tumour", "pituitary": "Pituitary"}
COLS = [("gradcam_pp", "Grad-CAM++"), ("ig", "IG"), ("smoothgrad", "SmoothGrad"),
        ("equal", "Equal Fusion"), ("agf_tau1", "AGF (τ = 1)"),
        ("agf_tausel", f"AGF (τ* = {TAU_STAR:g})")]

def _norm01(a):
    a = np.asarray(a, dtype=np.float64); r = a.max() - a.min()
    return (a - a.min()) / r if r > 0 else np.zeros_like(a)

spec = next(s for s in MODELS if s["key"] == BACKBONE)
model, eval_model, _, backbone_name, _ = load_model(spec)
target_layer = get_target_layer(model, backbone_name)

picks = {}                                         # first correctly classified per class
for idx in range(len(sample_paths)):
    y = int(sample_labels[idx])
    if y in picks:
        continue
    with torch.inference_mode():
        pred = int(eval_model(load_image_tensor(sample_paths[idx]).unsqueeze(0).to(DEVICE)).argmax(1).item())
    if pred == y:
        picks[y] = idx
    if len(picks) == len(class_names):
        break
print("Images:", {class_names[y]: i for y, i in sorted(picks.items())})

fig, axes = plt.subplots(len(picks), 1 + len(COLS), figsize=(7.2, 7.2 * len(picks) / 7 + 0.3),
                         squeeze=False)
for r, (y, idx) in enumerate(sorted(picks.items())):
    gray = np.array(RemoveBlackBorder()(Image.open(sample_paths[idx]).convert("RGB"))
                    .convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
    tensor = load_image_tensor(sample_paths[idx])
    indiv, scores, pred, _, _ = base_maps(eval_model, target_layer, tensor)
    maps, _ = method_maps(indiv, scores, {"agf_tau1": 1.0, "agf_tausel": TAU_STAR})
    panels = [("MRI", None)] + [(lab, maps[k]) for k, lab in COLS]
    for c, (lab, heat) in enumerate(panels):
        ax = axes[r, c]
        ax.imshow(gray, cmap="gray", interpolation="nearest")
        if heat is not None:
            ax.imshow(_norm01(heat), cmap="jet", alpha=0.45, interpolation="bilinear")
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_linewidth(0.5); s.set_color("#333333")
        if r == 0:
            ax.set_title(lab, fontsize=7)
    axes[r, 0].set_ylabel(ROW_LABEL.get(class_names[y], class_names[y]), fontsize=8, fontweight="bold")
    free_gpu()

fig.tight_layout(pad=0.3, w_pad=0.2, h_pad=0.2)
os.makedirs(os.path.join(OUTPUT_DIR, "Figures"), exist_ok=True)
for ext in ("pdf", "png"):
    fig.savefig(os.path.join(OUTPUT_DIR, "Figures", f"qualitative_grid.{ext}"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", os.path.join(OUTPUT_DIR, "Figures", "qualitative_grid.pdf"))


tau* = 0.25 from /kaggle/input/notebooks/tamannajarintima/agf-revision/revision/tau_selection.json
Images: {'glioma': 0, 'meningioma': 400, 'notumor': 800, 'pituitary': 1200}
Saved: /kaggle/working/Figures/qualitative_grid.pdf
